In [1]:
# CELL 20 — Full build setup: paths, chunk plan, and the v2 generator (identical to CELL 9; only paths, split rule and job list changed)
!pip install -q pikepdf pymupdf pypdf pebble

import os, io, re, glob, json, zlib, shutil, random, logging
import pandas as pd
import pikepdf, pypdf
import pymupdf as fitz
from pypdf import PdfWriter
from pypdf.generic import ArrayObject, StreamObject, NameObject
from PIL import Image, ImageDraw, ImageFont
from pebble import ProcessPool
from concurrent.futures import TimeoutError, as_completed
from tqdm.auto import tqdm

logging.getLogger("pypdf").setLevel(logging.ERROR)
logging.getLogger("pikepdf").setLevel(logging.ERROR)

# ---------- paths ----------
hits = sorted(glob.glob("/kaggle/input/**/clean_pool.csv", recursive=True))
assert hits, "clean_pool.csv not found - is PDP BPTP attached as input?"
CLEAN_POOL = hits[0]
SRC = os.path.join(os.path.dirname(CLEAN_POOL), "govdocs_pdfs")
assert os.path.isdir(SRC), f"govdocs_pdfs not found next to {CLEAN_POOL}"
WORK = "/kaggle/working/full"
OUT = f"{WORK}/tmp_pdfs"      # PDFs of the current chunk only (deleted after extraction)
META = f"{WORK}/meta"         # one small JSON per original (manifest rows)
CHUNKS = f"{WORK}/chunks"     # per-chunk results: features CSV + graphs file
for d in (OUT, META, CHUNKS):
    os.makedirs(d, exist_ok=True)

TIME_LIMIT = 180
CHUNK_SIZE = 300
WORKERS = 4
FAMILIES = ["acrobat distiller windows", "acrobat distiller for", "acrobat pdfwriter for",
            "adobe pdf library", "corel pdf engine"]
TOOLS = ["pikepdf", "pymupdf", "pypdf"]

LIT = re.compile(rb"\((?:[^()\\]|\\.)*\)", re.S)
TAIL = re.compile(rb"(?:\s+|-?(?:\d+\.?\d*|\.\d+)(?=[\s(\]])|\((?:[^()\\]|\\.)*\))*\s*(?:\]\s*TJ|Tj|'|\")")
VALUE = re.compile(rb"\d[\d,\.]*\d")

def make_stamp(text):
    img = Image.new("RGBA", (420, 140), (255, 255, 255, 0))
    dr = ImageDraw.Draw(img)
    dr.rectangle([4, 4, 415, 135], outline=(200, 0, 0, 255), width=8)
    try:
        font = ImageFont.truetype("DejaVuSans-Bold.ttf", 60)
    except Exception:
        font = ImageFont.load_default()
    dr.text((24, 34), text, fill=(200, 0, 0, 255), font=font)
    buf = io.BytesIO()
    img.save(buf, "PNG")
    return buf.getvalue()

PNG_HONEST = make_stamp("RECEIVED")
PNG_FAKE = make_stamp("PAID")

def digit_positions(seg):
    pos, i, n = [], 1, len(seg) - 1
    while i < n:
        b = seg[i]
        if b == 0x5C:
            j = i + 1
            if j < n and 48 <= seg[j] <= 55:
                k = j
                while k < n and k < j + 3 and 48 <= seg[k] <= 55:
                    k += 1
                i = k
            else:
                i = j + 1
            continue
        if 48 <= b <= 57:
            pos.append(i)
        i += 1
    return pos

def candidates(data):
    digits, values = [], []
    for m in LIT.finditer(data):
        if not TAIL.match(data[m.end():m.end() + 300]):
            continue
        seg = m.group()
        digits += [m.start() + p for p in digit_positions(seg)]
        for v in VALUE.finditer(seg):
            if v.start() > 0 and seg[v.start() - 1] == 0x5C:
                continue
            if sum(c in b"0123456789" for c in v.group()) >= 3:
                values.append((m.start() + v.start(), m.start() + v.end()))
    return digits, values

def new_value(old, rng):
    s = old.decode("latin-1")
    two_dec = s.count(".") == 1 and len(s.split(".")[1]) == 2
    intpart = s.split(".")[0] if two_dec else s
    nd = max(1, sum(c.isdigit() for c in intpart) + rng.choice([-1, 1, 2]))
    n = rng.randint(10 ** (nd - 1), 10 ** nd - 1)
    out = (f"{n:,}" if "," in s else str(n)) + (f".{rng.randint(0, 99):02d}" if two_dec else "")
    if out == s:
        out = str(n + 1)
    return out.encode()

def pk_streams(pdf, i):
    c = pdf.pages[i].obj.get("/Contents")
    if c is None:
        return []
    return [s for s in (list(c) if isinstance(c, pikepdf.Array) else [c]) if isinstance(s, pikepdf.Stream)]

def choose_specs(src, rng):
    dspec = vspec = None
    with pikepdf.open(src) as p:
        order = list(range(len(p.pages)))
        rng.shuffle(order)
        for i in order:
            for k, s in enumerate(pk_streams(p, i)):
                try:
                    data = s.read_bytes()
                except Exception:
                    continue
                d, v = candidates(data)
                if dspec is None and d:
                    pos = rng.choice(d)
                    old = data[pos:pos + 1]
                    new = bytes([rng.choice([c for c in b"0123456789" if c != old[0]])])
                    dspec = dict(page=i, k=k, start=pos, end=pos + 1, old=old, new=new)
                if vspec is None and v:
                    st, en = rng.choice(v)
                    vspec = dict(page=i, k=k, start=st, end=en, old=data[st:en], new=new_value(data[st:en], rng))
                if dspec and vspec:
                    return dspec, vspec
    return dspec, vspec

def apply_spec(data, spec):
    if data[spec["start"]:spec["end"]] != spec["old"]:
        raise RuntimeError("edit location mismatch")
    return data[:spec["start"]] + spec["new"] + data[spec["end"]:]

def edit_pikepdf(pdf, spec):
    s = pk_streams(pdf, spec["page"])[spec["k"]]
    s.write(apply_spec(s.read_bytes(), spec))

def edit_fitz(doc, spec):
    x = doc[spec["page"]].get_contents()[spec["k"]]
    doc.update_stream(x, apply_spec(doc.xref_stream(x), spec))

def py_streams(page):
    c = page.get("/Contents")
    if c is None:
        return []
    c = c.get_object()
    out = []
    for it in (list(c) if isinstance(c, ArrayObject) else [c]):
        o = it.get_object()
        if isinstance(o, StreamObject):
            out.append(o)
    return out

def edit_pypdf(w, spec):
    o = py_streams(w.pages[spec["page"]])[spec["k"]]
    new = apply_spec(o.get_data(), spec)
    flt = o.get("/Filter")
    if flt is not None and flt != "/FlateDecode" and list(flt) != ["/FlateDecode"]:
        o[NameObject("/Filter")] = NameObject("/FlateDecode")
    if "/DecodeParms" in o:
        del o["/DecodeParms"]
    o.set_data(new)

def run_tool(tool, src, out, spec, mode):
    if tool == "pikepdf":
        with pikepdf.open(src) as p:
            if spec:
                edit_pikepdf(p, spec)
            if mode == "recompress":
                p.save(out, compress_streams=True, recompress_flate=True,
                       object_stream_mode=pikepdf.ObjectStreamMode.generate)
            elif mode == "linearize":
                p.save(out, linearize=True)
            else:
                p.save(out)
    elif tool == "pymupdf":
        d = fitz.open(src)
        if spec:
            edit_fitz(d, spec)
        if mode == "recompress":
            for pg in d:
                for x in pg.get_contents():
                    d.update_stream(x, d.xref_stream(x))
            d.save(out, garbage=4, deflate=True)
        else:
            d.save(out)
        d.close()
    else:
        w = PdfWriter(clone_from=src)
        if spec:
            edit_pypdf(w, spec)
        if mode == "recompress":
            for pg in w.pages:
                pg.compress_content_streams()
        with open(out, "wb") as f:
            w.write(f)

def fitz_incr(src, out, action):
    shutil.copy(src, out)
    d = fitz.open(out)
    try:
        if d.is_repaired:
            raise RuntimeError("file needed repair")
        res = action(d)
        d.saveIncr()
        return res
    finally:
        d.close()

def fitz_full(src, out, action):
    d = fitz.open(src)
    try:
        res = action(d)
        d.save(out)
        return res
    finally:
        d.close()

def add_note(d, i):
    r = d[i].rect
    d[i].add_text_annot(fitz.Point(r.x0 + 36, r.y0 + 36), "Reviewed")

def add_stamp(d, i, png):
    r = d[i].rect
    d[i].insert_image(fitz.Rect(r.x1 - 210, r.y0 + 40, r.x1 - 40, r.y0 + 97), stream=png, overlay=True)

def add_textbox(d, i, text):
    r = d[i].rect
    d[i].add_freetext_annot(fitz.Rect(r.x0 + 40, r.y1 - 80, r.x0 + 300, r.y1 - 58), text, fontsize=11)

def cover_replace(d, rng):
    order = list(range(len(d)))
    rng.shuffle(order)
    for i in order:
        words = [w for w in d[i].get_text("words") if re.search(r"\d", w[4]) and len(w[4]) >= 2]
        if not words:
            continue
        x0, y0, x1, y1, txt = rng.choice(words)[:5]
        r = fitz.Rect(x0, y0, x1, y1)
        if r.is_empty or r.height < 3:
            continue
        new = "".join(rng.choice("0123456789") if c.isdigit() else c for c in txt)
        if new == txt:
            new = txt[:-1] + ("7" if txt[-1] != "7" else "3")
        d[i].draw_rect(r, color=None, fill=(1, 1, 1), overlay=True)
        d[i].insert_text(fitz.Point(x0, y1 - r.height * 0.2), new, fontsize=max(5, r.height * 0.8),
                         fontname="helv", color=(0, 0, 0))
        return {"page": i, "old": txt, "new": new}
    raise RuntimeError("no numeric word for cover-and-replace")

def cleanup(stem, keep_meta=False):
    for p in glob.glob(os.path.join(OUT, f"{stem}__*.pdf")):
        os.remove(p)
    mp = os.path.join(META, stem + ".json")
    if not keep_meta and os.path.exists(mp):
        os.remove(mp)

def split_of(name):
    return "dev" if zlib.crc32(name.encode()) % 100 < 15 else "train"

# ---------- one original -> 15 versions (same as CELL 9; only the split line differs) ----------
def process(job):
    name, fam = job
    fitz.TOOLS.mupdf_display_errors(False)
    stem = name[:-4]
    src = os.path.join(SRC, name)
    meta_path = os.path.join(META, stem + ".json")
    if os.path.exists(meta_path):
        with open(meta_path) as fh:
            rows = json.load(fh)
        if all(os.path.exists(os.path.join(OUT, r["out_file"])) for r in rows):
            return name, rows, None
    cleanup(stem)
    try:
        rng = random.Random(zlib.crc32(name.encode()) + 2026)
        split = split_of(name)
        t_main, t_comp = rng.choice(TOOLS), rng.choice(TOOLS)
        dspec, vspec = choose_specs(src, rng)
        if dspec is None or vspec is None:
            raise RuntimeError("no editable number")
        P = lambda v: os.path.join(OUT, f"{stem}__{v}.pdf")
        pg = dspec["page"]

        shutil.copy(src, P("G0_original"))
        run_tool(t_main, src, P("G1_resave"), None, "resave")
        run_tool(t_comp, src, P("G2_recompress"), None, "recompress")
        fitz_incr(src, P("G3_note_incr"), lambda d: add_note(d, pg))
        run_tool("pikepdf", src, P("G4_linearize"), None, "linearize")
        fitz_full(src, P("G5_stamp_honest"), lambda d: add_stamp(d, pg, PNG_HONEST))
        fitz_incr(src, P("G6_textbox_honest"), lambda d: add_textbox(d, pg, "Approved - HR dept."))

        run_tool(t_main, src, P("F1_digit_resave"), dspec, "resave")
        run_tool(t_comp, src, P("F2_digit_recompress"), dspec, "recompress")
        fitz_incr(src, P("F3_digit_incr"), lambda d: edit_fitz(d, dspec))
        run_tool(t_main, src, P("F4_value_resave"), vspec, "resave")
        cov = fitz_full(src, P("F5_cover_replace"), lambda d: cover_replace(d, rng))
        run_tool("pikepdf", src, P("F6_digit_linearize"), dspec, "linearize")
        fitz_full(src, P("F7_stamp_fake"), lambda d: add_stamp(d, pg, PNG_FAKE))
        fitz_incr(src, P("F8_textbox_fake"), lambda d: add_textbox(d, pg, "TOTAL DUE: $98,450.00"))

        dold, dnew = dspec["old"].decode("latin-1"), dspec["new"].decode("latin-1")
        vold, vnew = vspec["old"].decode("latin-1"), vspec["new"].decode("latin-1")
        spec = [
            ("G0_original", 0, "none", "original", "none", None, None, None, "G0"),
            ("G1_resave", 0, t_main, "full", "none", None, None, None, "G1"),
            ("G2_recompress", 0, t_comp, "full_recompress", "none", None, None, None, "G2"),
            ("G3_note_incr", 0, "pymupdf", "incremental", "note", pg, None, None, "G3"),
            ("G4_linearize", 0, "pikepdf", "linearize", "none", None, None, None, "G4"),
            ("G5_stamp_honest", 0, "pymupdf", "full", "stamp", pg, None, None, "G5"),
            ("G6_textbox_honest", 0, "pymupdf", "incremental", "textbox", pg, None, None, "G6"),
            ("F1_digit_resave", 1, t_main, "full", "digit", pg, dold, dnew, "G1"),
            ("F2_digit_recompress", 1, t_comp, "full_recompress", "digit", pg, dold, dnew, "G2"),
            ("F3_digit_incr", 1, "pymupdf", "incremental", "digit", pg, dold, dnew, "G3"),
            ("F4_value_resave", 1, t_main, "full", "value", vspec["page"], vold, vnew, "G1"),
            ("F5_cover_replace", 1, "pymupdf", "full", "cover", cov["page"], cov["old"], cov["new"], "G1"),
            ("F6_digit_linearize", 1, "pikepdf", "linearize", "digit", pg, dold, dnew, "G4"),
            ("F7_stamp_fake", 1, "pymupdf", "full", "stamp", pg, None, None, "G5"),
            ("F8_textbox_fake", 1, "pymupdf", "incremental", "textbox", pg, None, None, "G6"),
        ]
        rows = [{"out_file": f"{stem}__{v}.pdf", "orig_file": name, "family": fam, "variant": v,
                 "label": lab, "tool": tool, "save_mode": mode, "edit_type": et, "edit_page": ep,
                 "old_value": ov, "new_value": nv, "twin": tw, "split": split}
                for v, lab, tool, mode, et, ep, ov, nv, tw in spec]
        with open(meta_path, "w") as fh:
            json.dump(rows, fh)
        return name, rows, None
    except Exception as ex:
        cleanup(stem)
        return name, None, f"{type(ex).__name__}: {str(ex)[:50]}"

# ---------- job list and chunk plan ----------
pool = pd.read_csv(CLEAN_POOL)
pool = pool[pool.producer_family.isin(FAMILIES)].copy()
pool["fam_order"] = pool.producer_family.map({f: i for i, f in enumerate(FAMILIES)})
pool = pool.sort_values(["fam_order", "file"])
JOBS = list(zip(pool.file, pool.producer_family))
CHUNK_LIST = [JOBS[i:i + CHUNK_SIZE] for i in range(0, len(JOBS), CHUNK_SIZE)]

print("Originals dir:", SRC, "|", len(glob.glob(SRC + "/*.pdf")), "PDFs")
print("\n=== ORIGINALS PER FAMILY (all clean files, zips 000-019) ===")
print(pool.producer_family.value_counts().reindex(FAMILIES).to_string())
print("Total originals:", len(JOBS), "| expected files if all succeed:", len(JOBS) * 15)
print("Dev share (originals):", round(sum(split_of(n) == "dev" for n, _ in JOBS) / len(JOBS) * 100, 1), "%")
print("\nChunks:", len(CHUNK_LIST), "of up to", CHUNK_SIZE, "originals")
print("Chunks already finished:", len(glob.glob(f"{CHUNKS}/chunk_*_features.csv")))
print("\nLibrary versions -> pikepdf", pikepdf.__version__, "| pymupdf", fitz.VersionBind, "| pypdf", pypdf.__version__)
free = shutil.disk_usage("/kaggle/working").free / 1e9
print(f"Free disk in /kaggle/working: {free:.1f} GB")

Originals dir: /kaggle/input/notebooks/atharvagandhi/pdp-bptp/govdocs_pdfs | 5029 PDFs

=== ORIGINALS PER FAMILY (all clean files, zips 000-019) ===
producer_family
acrobat distiller windows    1566
acrobat distiller for         659
acrobat pdfwriter for         428
adobe pdf library             153
corel pdf engine              108
Total originals: 2914 | expected files if all succeed: 43710
Dev share (originals): 14.8 %

Chunks: 10 of up to 300 originals
Chunks already finished: 0

Library versions -> pikepdf 10.17.0 | pymupdf 2.0 | pypdf 6.21.2
Free disk in /kaggle/working: 20.9 GB


In [2]:
# CELL 21 — Feature extractor (exact CELL 10 code) + PDF->graph converter (exact fixed CELL 12 code), path-based; quick self-test
import os, re, zlib, math, gzip, pickle, logging, collections
import numpy as np
import pandas as pd
import pikepdf

logging.getLogger("pikepdf").setLevel(logging.ERROR)

FEAT_TIME_LIMIT = 60
GRAPH_TIME_LIMIT = 90
MAX_STREAMS = 120
TOOL_WORDS = [b"pypdf", b"pymupdf", b"mupdf", b"pikepdf", b"qpdf"]
STREAM_BODY = re.compile(rb"(?<!end)stream\r?\n.*?endstream", re.S)
STD14 = {"Helvetica", "Helvetica-Bold", "Helvetica-Oblique", "Helvetica-BoldOblique", "Times-Roman", "Times-Bold",
         "Times-Italic", "Times-BoldItalic", "Courier", "Courier-Bold", "Courier-Oblique", "Courier-BoldOblique",
         "Symbol", "ZapfDingbats"}

def ratio(a, b):
    return a / b if b else 0.0

# =====================================================================================
# PART A — document-level features (identical to CELL 10)
# =====================================================================================
def file_features(raw):
    f = {}
    skel = STREAM_BODY.sub(b"stream endstream", raw)
    f["size_log"] = math.log10(len(raw) + 1)
    m = re.match(rb"\s*%PDF-(\d)\.(\d)", raw[:64])
    f["version"] = int(m.group(1)) + int(m.group(2)) / 10 if m else 0
    f["binary_comment"] = int(bool(re.match(rb"\s*%PDF-\d\.\d\s*[\r\n]+%[\x80-\xff]{4}", raw[:40])))
    f["n_eof"] = raw.count(b"%%EOF")
    f["n_startxref"] = raw.count(b"startxref")
    f["xref_table"] = int(bool(re.search(rb"(^|[\r\n])xref\s*[\r\n]", skel)))
    f["xref_stream"] = int(bool(re.search(rb"/Type\s*/XRef", skel)))
    f["obj_streams"] = int(bool(re.search(rb"/Type\s*/ObjStm", skel)))
    f["linearized"] = int(b"/Linearized" in raw[:2048])
    f["true_revisions"] = f["n_eof"] - f["linearized"]
    crlf = skel.count(b"\r\n")
    cr = skel.count(b"\r") - crlf
    lf = skel.count(b"\n") - crlf
    tot = crlf + cr + lf
    f["eol_crlf"], f["eol_cr"], f["eol_lf"] = ratio(crlf, tot), ratio(cr, tot), ratio(lf, tot)
    objs = re.findall(rb"(\d+)\s+(\d+)\s+obj", skel)
    nums = [int(a) for a, _ in objs]
    f["n_obj_defs"] = len(nums)
    f["obj_order_increasing"] = ratio(sum(1 for a, b in zip(nums, nums[1:]) if b > a), max(len(nums) - 1, 1))
    f["obj_gen_nonzero"] = ratio(sum(1 for _, g in objs if g != b"0"), len(objs))
    f["obj_max_over_count"] = ratio(max(nums) if nums else 0, len(set(nums)))
    f["obj_redefined"] = len(nums) - len(set(nums))
    f["obj_newline_dict"] = ratio(len(re.findall(rb"obj\s*[\r\n]+\s*<<", skel)), len(nums))
    f["dict_open_space"] = ratio(skel.count(b"<< /"), skel.count(b"<<"))
    f["dict_close_space"] = ratio(skel.count(b" >>"), skel.count(b">>"))
    nstream = len(re.findall(rb"(?<!end)stream", skel))
    f["stream_kw_crlf"] = ratio(len(re.findall(rb"(?<!end)stream\r\n", raw)), nstream)
    f["endobj_newline_before"] = ratio(len(re.findall(rb"[\r\n]endobj", skel)), skel.count(b"endobj"))
    f["reals_dot_zero"] = ratio(len(re.findall(rb"\s\d+\.0[\s/\]>]", skel)), len(re.findall(rb"\s\d+\.\d+[\s/\]>]", skel)))
    f["has_ID"] = int(b"/ID" in skel)
    f["tool_word_in_file"] = int(any(w in skel.lower() for w in TOOL_WORDS))
    return f

def stream_sig(s):
    d = {}
    flt = s.get("/Filter")
    names = [str(x) for x in flt] if isinstance(flt, pikepdf.Array) else ([] if flt is None else [str(flt)])
    d["filter"] = "+".join(names) if names else "none"
    d["has_parms"] = int(s.get("/DecodeParms") is not None)
    d["len_indirect"] = int(bool(getattr(s.get("/Length"), "is_indirect", False)))
    d["hdr"], d["zlevel"], d["sync_end"], d["adler_ok"], d["tail_junk"], d["ratio"] = -1, 0, 0, 0, 0, 0.0
    raw = s.read_raw_bytes()
    if names == ["/FlateDecode"] and len(raw) >= 6:
        d["hdr"] = raw[0] * 256 + raw[1]
        try:
            dob = zlib.decompressobj()
            dec = dob.decompress(raw)
            body = raw[:len(raw) - len(dob.unused_data)]
            d["tail_junk"] = int(len(dob.unused_data) > 0)
            d["ratio"] = ratio(len(body), len(dec))
            d["adler_ok"] = int(body[-4:] == zlib.adler32(dec).to_bytes(4, "big"))
            d["sync_end"] = int(b"\x00\x00\xff\xff" in body[-12:])
            if len(dec) <= 2_000_000:
                for lv in range(1, 10):
                    if zlib.compress(dec, lv) == body:
                        d["zlevel"] = lv
                        break
        except Exception:
            d["hdr"] = -2
    return d

def resource_features(pdf):
    f = {"l5_n_annots": 0, "l5_n_freetext": 0, "l5_n_text_notes": 0, "l5_n_img_xobj": 0, "l5_n_fonts": 0,
         "l5_n_std14_unembedded": 0, "l5_frac_pages_multi_stream": 0.0, "l5_max_streams_per_page": 0}
    fonts_seen, imgs_seen, multi = set(), set(), 0
    for page in pdf.pages:
        po = page.obj
        annots = po.get("/Annots")
        if isinstance(annots, pikepdf.Array):
            for a in annots:
                try:
                    st = str(a.get("/Subtype", ""))
                except Exception:
                    continue
                f["l5_n_annots"] += 1
                f["l5_n_freetext"] += int(st == "/FreeText")
                f["l5_n_text_notes"] += int(st == "/Text")
        c = po.get("/Contents")
        ns = len(c) if isinstance(c, pikepdf.Array) else (1 if c is not None else 0)
        multi += int(ns > 1)
        f["l5_max_streams_per_page"] = max(f["l5_max_streams_per_page"], ns)
        try:
            res = page.resources
        except Exception:
            res = po.get("/Resources")
        if res is None:
            continue
        fdict = res.get("/Font")
        if isinstance(fdict, pikepdf.Dictionary):
            for _, fo in fdict.items():
                key = getattr(fo, "objgen", (0, 0))
                key = key if key != (0, 0) else id(fo)
                if key in fonts_seen:
                    continue
                fonts_seen.add(key)
                base = str(fo.get("/BaseFont", "")).lstrip("/")
                desc = fo.get("/FontDescriptor")
                embedded = desc is not None and any(k in desc for k in ("/FontFile", "/FontFile2", "/FontFile3"))
                if base in STD14 and not embedded and str(fo.get("/Subtype", "")) != "/Type0":
                    f["l5_n_std14_unembedded"] += 1
        xdict = res.get("/XObject")
        if isinstance(xdict, pikepdf.Dictionary):
            for _, xo in xdict.items():
                key = getattr(xo, "objgen", (0, 0))
                if key in imgs_seen:
                    continue
                try:
                    if str(xo.get("/Subtype", "")) == "/Image":
                        imgs_seen.add(key)
                except Exception:
                    pass
    f["l5_n_fonts"] = len(fonts_seen)
    f["l5_n_img_xobj"] = len(imgs_seen)
    f["l5_frac_pages_multi_stream"] = ratio(multi, len(pdf.pages))
    return f

def extract_features(path):
    with open(path, "rb") as fh:
        raw = fh.read()
    f = {"out_file": os.path.basename(path)}
    f.update(file_features(raw))
    with pikepdf.open(path) as pdf:
        info = pdf.docinfo
        prod = str(info.get("/Producer", "")) if info is not None else ""
        f["has_producer"] = int(bool(prod.strip()))
        f["producer_len"] = len(prod)
        cd, md = str(info.get("/CreationDate", "")), str(info.get("/ModDate", ""))
        f["moddate_eq_creation"] = int(bool(cd) and cd == md)
        f["has_xmp"] = int("/Metadata" in pdf.Root)
        f["n_pages"] = len(pdf.pages)
        try:
            f.update(resource_features(pdf))
        except Exception:
            pass
        streams = []
        for pi, page in enumerate(pdf.pages):
            c = page.obj.get("/Contents")
            if c is None:
                continue
            for s in (list(c) if isinstance(c, pikepdf.Array) else [c]):
                if isinstance(s, pikepdf.Stream):
                    streams.append((pi, s))
            if len(streams) >= MAX_STREAMS:
                break
        streams = streams[:MAX_STREAMS]
        sigs = []
        for pi, s in streams:
            try:
                sigs.append((pi, getattr(s, "objgen", (-1, 0))[0], stream_sig(s)))
            except Exception:
                pass
        cs_nums = {num for _, num, _ in sigs}

    n = len(sigs)
    f["cs_n"] = n
    if n:
        fl = [d["filter"] for _, _, d in sigs]
        f["cs_frac_flate"] = ratio(fl.count("/FlateDecode"), n)
        f["cs_frac_lzw"] = ratio(sum("LZW" in x for x in fl), n)
        f["cs_frac_none"] = ratio(fl.count("none"), n)
        f["cs_n_filters"] = len(set(fl))
        flate = [d for _, _, d in sigs if d["filter"] == "/FlateDecode"]
        f["cs_frac_zlib_repro"] = ratio(sum(d["zlevel"] > 0 for d in flate), len(flate)) if flate else np.nan
        f["cs_n_hdr"] = len({d["hdr"] for _, _, d in sigs})
        f["cs_n_zlevel"] = len({d["zlevel"] for _, _, d in sigs})
        f["cs_frac_sync"] = np.mean([d["sync_end"] for _, _, d in sigs])
        f["cs_frac_adler"] = np.mean([d["adler_ok"] for _, _, d in sigs])
        f["cs_frac_junk"] = np.mean([d["tail_junk"] for _, _, d in sigs])
        f["cs_frac_len_indirect"] = np.mean([d["len_indirect"] for _, _, d in sigs])
        r = np.array([d["ratio"] for _, _, d in sigs])
        f["cs_ratio_std"] = float(r.std())
        f["cs_ratio_max_z"] = float(np.max(np.abs(r - r.mean())) / r.std()) if r.std() > 0 else 0.0
        keys = [(d["filter"], d["hdr"], d["zlevel"], d["sync_end"], d["adler_ok"], d["tail_junk"], d["has_parms"], d["len_indirect"]) for _, _, d in sigs]
        cnt = collections.Counter(keys)
        f["cs_n_sig"] = len(cnt)
        f["cs_majority_share"] = max(cnt.values()) / n
        f["cs_n_singletons"] = sum(1 for v in cnt.values() if v == 1) if n >= 3 else 0
        odd = [pi for (pi, _, _), k in zip(sigs, keys) if cnt[k] == 1] if n >= 3 else []
        f["loc_page"] = odd[0] if len(odd) == 1 else -1

    f.update({"upd_n_objs": 0, "upd_touch_content": 0, "upd_has_annot": 0, "upd_has_page": 0, "upd_has_sig": 0})
    if f["true_revisions"] > 1:
        ends = [m.end() for m in re.finditer(rb"%%EOF", raw)]
        tail = STREAM_BODY.sub(b"stream endstream", raw[ends[-2]:])
        objs = re.findall(rb"(\d+)\s+\d+\s+obj(.*?)endobj", tail, re.S)
        f["upd_n_objs"] = len(objs)
        for num, body in objs:
            if int(num) in cs_nums:
                f["upd_touch_content"] = 1
            if re.search(rb"/Type\s*/Annot|/Subtype\s*/(Text|FreeText)\b", body):
                f["upd_has_annot"] = 1
            if re.search(rb"/Type\s*/Page\b", body):
                f["upd_has_page"] = 1
            if b"/ByteRange" in body:
                f["upd_has_sig"] = 1
    return f

# =====================================================================================
# PART B — PDF -> graph (identical to fixed CELL 12)
# =====================================================================================
NODE_TYPES = ["catalog", "pages", "page", "content", "font", "fontdesc", "fontfile", "image", "form", "annot",
              "metadata", "xref", "extgstate", "other_dict", "array", "other_stream", "other"]
EDGE_KEYS = ["/Parent", "/Kids", "/Contents", "/Resources", "/Font", "/XObject", "/Annots", "/FontDescriptor",
             "/FontFile", "/Metadata", "/Pages", "other"]
HDRS = [0x789C, 0x78DA, 0x7801, 0x785E]
FEAT_NAMES = (["t_" + t for t in NODE_TYPES] + ["is_stream", "raw_len_log", "n_keys_log"] +
              ["f_flate", "f_lzw", "f_dct", "f_none", "f_other"] +
              ["h_789C", "h_78DA", "h_7801", "h_785E", "h_other"] +
              ["zlevel", "zlib_repro", "adler_ok", "sync_end", "tail_junk", "ratio",
               "objnum_rank", "gen_nonzero", "std14_unembedded", "annot_freetext", "annot_text",
               "len_indirect", "entropy"])
D = len(FEAT_NAMES)
LEN_REF = re.compile(rb"/Length\s+\d+\s+\d+\s+R")

def edge_key(k):
    if k is None:
        return "other"
    if k.startswith("/FontFile"):
        return "/FontFile"
    return k if k in EDGE_KEYS else "other"

def collect_refs(o, carried, out, depth=0):
    if depth > 8:
        return
    try:
        if isinstance(o, (pikepdf.Dictionary, pikepdf.Stream)):
            for k, v in o.items():
                kk = k if (k in EDGE_KEYS or k.startswith("/FontFile")) else carried
                if getattr(v, "is_indirect", False):
                    out.append((edge_key(kk), v.objgen))
                else:
                    collect_refs(v, kk, out, depth + 1)
        elif isinstance(o, pikepdf.Array):
            for v in o:
                if getattr(v, "is_indirect", False):
                    out.append((edge_key(carried), v.objgen))
                else:
                    collect_refs(v, carried, out, depth + 1)
    except Exception:
        pass

def node_type(o, content_set):
    if isinstance(o, pikepdf.Stream):
        if o.objgen in content_set:
            return "content"
        st, ty = str(o.get("/Subtype", "")), str(o.get("/Type", ""))
        if st == "/Image":
            return "image"
        if st == "/Form":
            return "form"
        if ty == "/Metadata":
            return "metadata"
        if ty in ("/XRef", "/ObjStm"):
            return "xref"
        if "/Length1" in o or st in ("/Type1C", "/CIDFontType0C", "/OpenType"):
            return "fontfile"
        return "other_stream"
    if isinstance(o, pikepdf.Dictionary):
        ty, st = str(o.get("/Type", "")), str(o.get("/Subtype", ""))
        mapping = {"/Catalog": "catalog", "/Pages": "pages", "/Page": "page", "/Font": "font",
                   "/FontDescriptor": "fontdesc", "/Annot": "annot", "/ExtGState": "extgstate"}
        if ty in mapping:
            return mapping[ty]
        if st in ("/Text", "/FreeText", "/Link", "/Widget", "/Stamp", "/Square", "/Highlight"):
            return "annot"
        return "other_dict"
    if isinstance(o, pikepdf.Array):
        return "array"
    return "other"

def shannon(raw):
    if not raw:
        return 0.0
    b = np.frombuffer(raw[:2_000_000], dtype=np.uint8)
    c = np.bincount(b, minlength=256)
    p = c[c > 0] / len(b)
    return float(-(p * np.log2(p)).sum() / 8)

def stream_feats(o, is_content):
    v = {"raw_len_log": 0.0, "f": "f_none", "h": None, "zlevel": 0, "zlib_repro": 0, "adler_ok": 0,
         "sync_end": 0, "tail_junk": 0, "ratio": 0.0, "len_indirect": 0, "entropy": 0.0}
    try:
        v["len_indirect"] = int(bool(LEN_REF.search(o.stream_dict.unparse())))
    except Exception:
        pass
    flt = o.get("/Filter")
    names = [str(x) for x in flt] if isinstance(flt, pikepdf.Array) else ([] if flt is None else [str(flt)])
    raw = o.read_raw_bytes()
    v["raw_len_log"] = math.log10(len(raw) + 1) / 7
    v["entropy"] = shannon(raw)
    if not names:
        v["f"] = "f_none"
    elif names == ["/FlateDecode"]:
        v["f"] = "f_flate"
    elif any("LZW" in n for n in names):
        v["f"] = "f_lzw"
    elif any("DCT" in n for n in names):
        v["f"] = "f_dct"
    else:
        v["f"] = "f_other"
    if v["f"] == "f_flate" and len(raw) >= 6:
        hdr = raw[0] * 256 + raw[1]
        v["h"] = "h_%04X" % hdr if hdr in HDRS else "h_other"
        if is_content or len(raw) <= 200_000:
            try:
                dob = zlib.decompressobj()
                dec = dob.decompress(raw)
                body = raw[:len(raw) - len(dob.unused_data)]
                v["tail_junk"] = int(len(dob.unused_data) > 0)
                v["ratio"] = min(len(body) / max(len(dec), 1), 1.5)
                v["adler_ok"] = int(body[-4:] == zlib.adler32(dec).to_bytes(4, "big"))
                v["sync_end"] = int(b"\x00\x00\xff\xff" in body[-12:])
                if len(dec) <= 2_000_000:
                    for lv in range(1, 10):
                        if zlib.compress(dec, lv) == body:
                            v["zlevel"], v["zlib_repro"] = lv, 1
                            break
            except Exception:
                pass
    return v

def to_graph(path):
    with pikepdf.open(path) as pdf:
        content_page = {}
        for pi, page in enumerate(pdf.pages):
            c = page.obj.get("/Contents")
            if c is None:
                continue
            for s in (list(c) if isinstance(c, pikepdf.Array) else [c]):
                if isinstance(s, pikepdf.Stream) and s.objgen not in content_page:
                    content_page[s.objgen] = pi
        objs = [o for o in pdf.objects if hasattr(o, "objgen") and o.objgen != (0, 0)]
        idx = {o.objgen: i for i, o in enumerate(objs)}
        max_num = max([g[0] for g in idx] + [1])
        X = np.zeros((len(objs), D), dtype=np.float16)
        node_page = np.full(len(objs), -1, dtype=np.int16)
        col = {n: j for j, n in enumerate(FEAT_NAMES)}
        src, dst, et = [], [], []
        for i, o in enumerate(objs):
            t = node_type(o, content_page)
            X[i, col["t_" + t]] = 1
            X[i, col["objnum_rank"]] = o.objgen[0] / max_num
            X[i, col["gen_nonzero"]] = int(o.objgen[1] != 0)
            if o.objgen in content_page:
                node_page[i] = content_page[o.objgen]
            try:
                if isinstance(o, (pikepdf.Dictionary, pikepdf.Stream)):
                    X[i, col["n_keys_log"]] = math.log10(len(list(o.keys())) + 1)
                if isinstance(o, pikepdf.Stream):
                    X[i, col["is_stream"]] = 1
                    sf = stream_feats(o, t == "content")
                    X[i, col["raw_len_log"]] = sf["raw_len_log"]
                    X[i, col[sf["f"]]] = 1
                    if sf["h"]:
                        X[i, col[sf["h"]]] = 1
                    X[i, col["zlevel"]] = sf["zlevel"] / 9
                    for k in ["zlib_repro", "adler_ok", "sync_end", "tail_junk", "ratio", "len_indirect", "entropy"]:
                        X[i, col[k]] = sf[k]
                if t == "font":
                    base = str(o.get("/BaseFont", "")).lstrip("/")
                    desc = o.get("/FontDescriptor")
                    emb = desc is not None and any(k in desc for k in ("/FontFile", "/FontFile2", "/FontFile3"))
                    X[i, col["std14_unembedded"]] = int(base in STD14 and not emb and str(o.get("/Subtype", "")) != "/Type0")
                if t == "annot":
                    st = str(o.get("/Subtype", ""))
                    X[i, col["annot_freetext"]] = int(st == "/FreeText")
                    X[i, col["annot_text"]] = int(st == "/Text")
            except Exception:
                pass
            refs = []
            collect_refs(o, None, refs)
            for k, tgt in refs:
                if tgt in idx:
                    src.append(i)
                    dst.append(idx[tgt])
                    et.append(EDGE_KEYS.index(k))
    return {"out_file": os.path.basename(path), "x": X, "edge_index": np.array([src, dst], dtype=np.int32),
            "edge_type": np.array(et, dtype=np.int8), "node_page": node_page}

# =====================================================================================
# Quick self-test on 3 original PDFs (reads only, writes nothing)
# =====================================================================================
for name, fam in [JOBS[0], JOBS[len(JOBS) // 2], JOBS[-1]]:
    p = os.path.join(SRC, name)
    fe = extract_features(p)
    gr = to_graph(p)
    print(f"{name} ({fam}): {len(fe) - 1} feature values | graph {gr['x'].shape[0]} nodes, "
          f"{gr['edge_index'].shape[1]} edges, {gr['x'].shape[1]} node features")
print("\nCELL 21 ready. Expected: 62 feature values (61 if a file has no page streams), 43 node features.")

000010.pdf (acrobat distiller windows): 62 feature values | graph 45 nodes, 60 edges, 43 node features
018877.pdf (acrobat distiller windows): 62 feature values | graph 183 nodes, 737 edges, 43 node features
019688.pdf (corel pdf engine): 62 feature values | graph 216 nodes, 538 edges, 43 node features

CELL 21 ready. Expected: 62 feature values (61 if a file has no page streams), 43 node features.


In [3]:
# CELL 27 — Attack setup: pick documents (403 dev + 600 train), recreate each document's exact F1 digit edit, verify it matches the full build
import os, glob, zlib, random, pickle
import pandas as pd
from pebble import ProcessPool
from concurrent.futures import TimeoutError, as_completed
from tqdm.auto import tqdm

hf = sorted(glob.glob("/kaggle/input/**/full/full_features.csv", recursive=True))
assert hf, "full_features.csv not found - is BTP Full attached as input?"
ATK = "/kaggle/working/attack"
os.makedirs(ATK, exist_ok=True)
feat_full = pd.read_csv(hf[0])
f1 = feat_full[feat_full.variant == "F1_digit_resave"].set_index("orig_file")
print("Full build features:", hf[0], "| F1 rows:", len(f1))

dev_docs = sorted(f1.index[f1.split == "dev"])
train_docs = sorted(f1.index[f1.split == "train"])
train_pick = sorted(random.Random(7).sample(train_docs, 600))
DOCS = [(d, "dev") for d in dev_docs] + [(d, "train") for d in train_pick]
print(f"Documents to attack: {len(dev_docs)} dev + {len(train_pick)} train = {len(DOCS)}")

def recreate(name):
    src = os.path.join(SRC, name)
    rng = random.Random(zlib.crc32(name.encode()) + 2026)
    t_main, t_comp = rng.choice(TOOLS), rng.choice(TOOLS)
    dspec, vspec = choose_specs(src, rng)
    return name, t_main, dspec

def norm(x):
    s = str(x)
    return s[:-2] if s.endswith(".0") else s

SPECS, problems = {}, []
with ProcessPool(max_workers=WORKERS) as pool_ex:
    futs = {pool_ex.schedule(recreate, args=(d,), timeout=120): (d, sp) for d, sp in DOCS}
    for fu in tqdm(as_completed(futs), total=len(futs), desc="Recreating F1 edits"):
        d, sp = futs[fu]
        try:
            name, t_main, dspec = fu.result()
        except TimeoutError:
            problems.append((d, "timeout"))
            continue
        except Exception as ex:
            problems.append((d, f"{type(ex).__name__}: {str(ex)[:50]}"))
            continue
        row = f1.loc[d]
        if dspec is None:
            problems.append((d, "no edit found"))
            continue
        ok = (row.tool == t_main and int(row.edit_page) == dspec["page"] and
              norm(row.old_value) == dspec["old"].decode("latin-1") and norm(row.new_value) == dspec["new"].decode("latin-1"))
        if not ok:
            problems.append((d, f"mismatch: build=({row.tool},{row.edit_page},{row.old_value}->{row.new_value}) "
                                f"now=({t_main},{dspec['page']},{dspec['old']}->{dspec['new']})"))
            continue
        SPECS[d] = {"split": sp, "family": row.family, "tool": t_main, "dspec": dspec}

with open(f"{ATK}/specs.pkl", "wb") as fh:
    pickle.dump(SPECS, fh)

ok_df = pd.DataFrame([{"doc": d, **{k: v for k, v in s.items() if k != "dspec"}} for d, s in SPECS.items()])
print(f"\nRecreated and verified: {len(SPECS)} of {len(DOCS)} ({len(SPECS) / len(DOCS) * 100:.1f}%)")
print("\nVerified documents per family and split:")
print(pd.crosstab(ok_df.family, ok_df.split).to_string())
print("\nTools used for the F1 edit:", ok_df.tool.value_counts().to_dict())
print("\nProblems:", len(problems))
for d, why in problems[:8]:
    print("  ", d, "->", why)
print(f"\nSaved: {ATK}/specs.pkl")

Full build features: /kaggle/input/notebooks/atharvagandhi/btp-full/full/full_features.csv | F1 rows: 2722
Documents to attack: 403 dev + 600 train = 1003


Recreating F1 edits:   0%|          | 0/1003 [00:00<?, ?it/s]


Recreated and verified: 1003 of 1003 (100.0%)

Verified documents per family and split:
split                      dev  train
family                               
acrobat distiller for       87    142
acrobat distiller windows  220    325
acrobat pdfwriter for       62     95
adobe pdf library           17     13
corel pdf engine            17     25

Tools used for the F1 edit: {'pypdf': 351, 'pymupdf': 329, 'pikepdf': 323}

Problems: 0

Saved: /kaggle/working/attack/specs.pkl


In [5]:
# CELL 28 (fixed) — Build 6 attacked versions of each F1 fake, verify edit visible + decoys present, extract features (fast mimic, real decoys, clean trace hiding, resumable)
import os, re, zlib, pickle, random, collections
import numpy as np
import pandas as pd
import pikepdf
import pymupdf as fitz
from pebble import ProcessPool
from concurrent.futures import TimeoutError, as_completed
from tqdm.auto import tqdm

ATK = "/kaggle/working/attack"
ATK_PDF = f"{ATK}/tmp_pdfs"
ATK_FEAT = f"{ATK}/attack_features_v2.csv"
ATK_FAIL = f"{ATK}/attack_failures_v2.csv"
os.makedirs(ATK_PDF, exist_ok=True)
for p in os.listdir(ATK_PDF):
    os.remove(os.path.join(ATK_PDF, p))
with open(f"{ATK}/specs.pkl", "rb") as fh:
    SPECS = pickle.load(fh)

ATTACKS = ["A0_plain", "A2a_mimic_simple", "A2b_mimic_aware", "A3_hide_traces", "A4_decoys", "A5c_combined"]
TIME_LIMIT = 240
SYNC = b"\x00\x00\xff\xff"
LEVELS_FOR_HDR = {0x7801: [1], 0x785E: [2, 3, 4, 5], 0x789C: [6], 0x78DA: [7, 8, 9]}
STRATS = [zlib.Z_DEFAULT_STRATEGY, zlib.Z_FILTERED, zlib.Z_RLE, zlib.Z_HUFFMAN_ONLY]

def sig_of(raw, std=None):
    d = {"hdr": -1, "zlevel": 0, "sync_end": 0, "adler_ok": 0, "tail_junk": 0, "ratio": 0.0}
    if len(raw) < 6:
        return d
    d["hdr"] = raw[0] * 256 + raw[1]
    try:
        dob = zlib.decompressobj()
        dec = dob.decompress(raw)
        body = raw[:len(raw) - len(dob.unused_data)]
        d["tail_junk"] = int(len(dob.unused_data) > 0)
        d["ratio"] = len(body) / max(len(dec), 1)
        d["adler_ok"] = int(body[-4:] == zlib.adler32(dec).to_bytes(4, "big"))
        d["sync_end"] = int(SYNC in body[-12:])
        if len(dec) <= 2_000_000:
            std = std or {lv: zlib.compress(dec, lv) for lv in range(1, 10)}
            d["zlevel"] = next((lv for lv, b in std.items() if b == body), 0)
    except Exception:
        pass
    return d

def page_streams(pdf, i):
    c = pdf.pages[i].obj.get("/Contents")
    if c is None:
        return []
    return [s for s in (list(c) if isinstance(c, pikepdf.Array) else [c]) if isinstance(s, pikepdf.Stream)]

def target_sig(pdf, skip):
    sigs = []
    for i in range(len(pdf.pages)):
        for s in page_streams(pdf, i):
            if s.objgen == skip or str(s.get("/Filter", "")) != "/FlateDecode":
                continue
            try:
                sigs.append(sig_of(s.read_raw_bytes()))
            except Exception:
                pass
    if not sigs:
        return None
    key = lambda d: (d["hdr"], d["zlevel"], d["sync_end"], d["adler_ok"], d["tail_junk"])
    best = collections.Counter(key(d) for d in sigs).most_common(1)[0][0]
    return dict(zip(["hdr", "zlevel", "sync_end", "adler_ok", "tail_junk"], best),
                ratio=float(np.median([d["ratio"] for d in sigs])))

def mimic(data, tgt):
    """Compress `data` so its measured signature matches tgt as well as possible. Returns (bytes, score of 6)."""
    std = {lv: zlib.compress(data, lv) for lv in range(1, 10)}
    if tgt["zlevel"] > 0:                       # target is reproducible by standard zlib (e.g. Corel)
        body = std[tgt["zlevel"]]
        cands = [body]
    else:
        cands = []
        for lv in LEVELS_FOR_HDR.get(tgt["hdr"], [6]):
            for st in STRATS:
                for mem in (8, 9):
                    co = zlib.compressobj(lv, zlib.DEFLATED, 15, mem, st)
                    if tgt["sync_end"]:
                        b = co.compress(data) + co.flush(zlib.Z_SYNC_FLUSH) + co.flush(zlib.Z_FINISH)
                    else:
                        b = co.compress(data) + co.flush()
                    cands.append(b)
    best, best_score = None, -1
    for b in cands:
        if tgt["tail_junk"]:
            b = b + b"\n"
        s = sig_of(b, std)
        score = sum([s["hdr"] == tgt["hdr"], s["zlevel"] == tgt["zlevel"], s["sync_end"] == tgt["sync_end"],
                     s["adler_ok"] == tgt["adler_ok"], s["tail_junk"] == tgt["tail_junk"],
                     abs(s["ratio"] - tgt["ratio"]) < 0.15])
        if score > best_score:
            best, best_score = b, score
    return best, best_score

def decoy_bytes(rng):
    lines = [b"q Q"] + [b"% " + bytes(rng.choice(b"abcdefghijklmnopqrstuvwxyz0123456789 ") for _ in range(rng.randint(40, 70)))
                        for _ in range(rng.randint(5, 15))]
    return b"\n".join(lines) + b"\n"

def add_decoys(pdf, page_i, rng, compress=None):
    po = pdf.pages[page_i].obj
    c = po.get("/Contents")
    arr = list(c) if isinstance(c, pikepdf.Array) else ([c] if c is not None else [])
    n = rng.randint(2, 4)
    for _ in range(n):
        raw = decoy_bytes(rng)
        if compress is None:
            st = pikepdf.Stream(pdf, raw)                       # left uncompressed -> saved with the same compressor as the edited stream
        else:
            st = pikepdf.Stream(pdf, compress(raw))
            st.stream_dict[pikepdf.Name.Filter] = pikepdf.Name.FlateDecode
        arr.append(pdf.make_indirect(st))
    po[pikepdf.Name.Contents] = pikepdf.Array(arr)
    return n

def strip_tool_words(raw):
    for w in [b"pypdf", b"PyPDF", b"pymupdf", b"PyMuPDF", b"mupdf", b"MuPDF", b"pikepdf", b"qpdf", b"QPDF"]:
        raw = raw.replace(w, (b"Adobe" + b" " * 10)[:len(w)])   # same length -> file offsets stay valid
    return raw

def build_one(src, dspec, attack, rng, info0, id0):
    out = os.path.join(ATK_PDF, f"{os.path.basename(src)[:-4]}__{attack}.pdf")
    n_decoys, mimic_score = 0, None
    with pikepdf.open(src) as pdf:
        es = page_streams(pdf, dspec["page"])[dspec["k"]]
        data = es.read_bytes()
        if data[dspec["start"]:dspec["end"]] != dspec["old"]:
            raise RuntimeError("edit location mismatch")
        new = data[:dspec["start"]] + dspec["new"] + data[dspec["end"]:]
        tgt = target_sig(pdf, es.objgen)
        if attack in ("A0_plain", "A3_hide_traces", "A4_decoys") or tgt is None:
            es.write(new)
        elif attack == "A2a_mimic_simple":
            lv = tgt["zlevel"] if tgt["zlevel"] > 0 else LEVELS_FOR_HDR.get(tgt["hdr"], [6])[0]
            es.write(zlib.compress(new, lv), filter=pikepdf.Name.FlateDecode)
        else:  # A2b_mimic_aware, A5c_combined
            body, mimic_score = mimic(new, tgt)
            es.write(body, filter=pikepdf.Name.FlateDecode)
        if attack == "A4_decoys":
            n_decoys = add_decoys(pdf, dspec["page"], rng)
        if attack == "A5c_combined":
            n_decoys = add_decoys(pdf, dspec["page"], rng, compress=(lambda b: mimic(b, tgt)[0]) if tgt else None)
        if attack in ("A3_hide_traces", "A5c_combined"):
            for k, v in info0.items():
                if v is not None:
                    pdf.docinfo[pikepdf.Name(k)] = v
            if id0 is not None:
                pdf.trailer[pikepdf.Name.ID] = id0
        pdf.save(out)
    if attack in ("A3_hide_traces", "A5c_combined"):
        with open(out, "rb") as fh:
            raw = fh.read()
        with open(out, "wb") as fh:
            fh.write(strip_tool_words(raw))
    return out, n_decoys, mimic_score, tgt is not None

def build(job):
    name, spec = job
    fitz.TOOLS.mupdf_display_errors(False)
    src = os.path.join(SRC, name)
    dspec = spec["dspec"]
    with pikepdf.open(src) as p0:
        info0 = {k: p0.docinfo.get(k) for k in ("/Producer", "/Creator", "/CreationDate", "/ModDate")}
        id0 = p0.trailer.get("/ID")
        n_streams0 = len(page_streams(p0, dspec["page"]))
    with fitz.open(src) as d0:
        text0 = d0[dspec["page"]].get_text()
    rows, errs = [], []
    for attack in ATTACKS:
        rng = random.Random(zlib.crc32((name + attack).encode()))
        try:
            out, n_dec, ms, has_tgt = build_one(src, dspec, attack, rng, info0, id0)
            with fitz.open(out) as d:
                visible = int(d[dspec["page"]].get_text() != text0)
            with pikepdf.open(out) as p1:
                n_streams1 = len(page_streams(p1, dspec["page"]))
            f = extract_features(out)
            f.update({"doc": name, "attack": attack, "family": spec["family"], "split": spec["split"], "tool": spec["tool"],
                      "edit_visible": visible, "decoys_planned": n_dec, "decoys_found": n_streams1 - n_streams0,
                      "mimic_score": ms, "has_siblings": int(has_tgt)})
            rows.append(f)
        except Exception as ex:
            errs.append({"doc": name, "attack": attack, "reason": f"{type(ex).__name__}: {str(ex)[:60]}"})
        finally:
            p = os.path.join(ATK_PDF, f"{name[:-4]}__{attack}.pdf")
            if os.path.exists(p):
                os.remove(p)
    return rows, errs

# ---------- resume: skip documents already fully done ----------
prev = pd.read_csv(ATK_FEAT) if os.path.exists(ATK_FEAT) else pd.DataFrame()
done_docs = set(prev.groupby("doc").attack.nunique().loc[lambda s: s == len(ATTACKS)].index) if len(prev) else set()
jobs = [(d, s) for d, s in SPECS.items() if d not in done_docs]
print("Documents:", len(SPECS), "| already done:", len(done_docs), "| to build now:", len(jobs))

rows = prev.to_dict("records") if len(prev) else []
fails = []
with ProcessPool(max_workers=WORKERS) as pool_ex:
    futs = {pool_ex.schedule(build, args=(j,), timeout=TIME_LIMIT): j[0] for j in jobs}
    for i, fu in enumerate(tqdm(as_completed(futs), total=len(futs), desc="Building attacks")):
        name = futs[fu]
        try:
            r, e = fu.result()
            rows += r
            fails += e
        except TimeoutError:
            fails.append({"doc": name, "attack": "all", "reason": "Timeout"})
        except Exception as ex:
            fails.append({"doc": name, "attack": "all", "reason": f"{type(ex).__name__}: {str(ex)[:60]}"})
        if (i + 1) % 100 == 0:
            pd.DataFrame(rows).to_csv(ATK_FEAT, index=False)

feat = pd.DataFrame(rows)
feat.to_csv(ATK_FEAT, index=False)
pd.DataFrame(fails).to_csv(ATK_FAIL, index=False)

print("\nAttacked files with features:", len(feat), "| documents:", feat.doc.nunique(), "| failures:", len(fails))
if fails:
    print("Failure reasons:", pd.Series([f"{x['attack']}: {x['reason'].split(':')[0]}" for x in fails]).value_counts().head(8).to_dict())
print("\nFiles per attack:")
print(feat.attack.value_counts().reindex(ATTACKS).to_string())
print("\n=== CHECKS PER ATTACK ===")
chk = feat.groupby("attack").agg(edit_visible_pct=("edit_visible", lambda s: round(s.mean() * 100, 1)),
                                 decoys_planned=("decoys_planned", "mean"), decoys_found=("decoys_found", "mean"),
                                 docs_with_sibling_streams_pct=("has_siblings", lambda s: round(s.mean() * 100, 1)))
print(chk.reindex(ATTACKS).round(2).to_string())
ms = feat[feat.attack == "A2b_mimic_aware"].mimic_score.dropna()
print("\n=== A2b feature-aware mimic: measured features matched (of 6) ===")
print("mean:", round(ms.mean(), 2), "| all 6:", round((ms == 6).mean() * 100, 1), "% | >=5:", round((ms >= 5).mean() * 100, 1), "%")
print(feat[feat.attack == "A2b_mimic_aware"].groupby("family").mimic_score.mean().round(2).to_string())
print(f"\nSaved: {ATK_FEAT}")

Documents: 1003 | already done: 0 | to build now: 1003


Building attacks:   0%|          | 0/1003 [00:00<?, ?it/s]


Attacked files with features: 4030 | documents: 1003 | failures: 1988
Failure reasons: {'A3_hide_traces: ForeignObjectError': 994, 'A5c_combined: ForeignObjectError': 994}

Files per attack:
attack
A0_plain            1003
A2a_mimic_simple    1003
A2b_mimic_aware     1003
A3_hide_traces         9
A4_decoys           1003
A5c_combined           9

=== CHECKS PER ATTACK ===
                  edit_visible_pct  decoys_planned  decoys_found  docs_with_sibling_streams_pct
attack                                                                                         
A0_plain                      99.6            0.00          0.00                           85.2
A2a_mimic_simple              99.6            0.00          0.00                           85.2
A2b_mimic_aware               99.6            0.00          0.00                           85.2
A3_hide_traces               100.0            0.00          0.00                            0.0
A4_decoys                     99.6            3.

In [6]:
# CELL 28b — Fix A3/A5c (copy metadata as plain bytes), build only missing attack files, then diagnose which feature the A2b mimic cannot match
import os, zlib, random, collections
import numpy as np
import pandas as pd
import pikepdf
import pymupdf as fitz
from pebble import ProcessPool
from concurrent.futures import TimeoutError, as_completed
from tqdm.auto import tqdm

def read_meta_plain(src):
    with pikepdf.open(src) as p0:
        info = {}
        for k in ("/Producer", "/Creator", "/CreationDate", "/ModDate"):
            v = p0.docinfo.get(k) if p0.docinfo is not None else None
            if isinstance(v, pikepdf.String):
                info[k] = bytes(v)
        idv = p0.trailer.get("/ID")
        ids = [bytes(x) for x in idv] if isinstance(idv, pikepdf.Array) else None
    return info, ids

def build_one_fixed(src, dspec, attack, rng, info_b, ids_b):
    out = os.path.join(ATK_PDF, f"{os.path.basename(src)[:-4]}__{attack}.pdf")
    n_decoys, mimic_score = 0, None
    with pikepdf.open(src) as pdf:
        es = page_streams(pdf, dspec["page"])[dspec["k"]]
        data = es.read_bytes()
        if data[dspec["start"]:dspec["end"]] != dspec["old"]:
            raise RuntimeError("edit location mismatch")
        new = data[:dspec["start"]] + dspec["new"] + data[dspec["end"]:]
        tgt = target_sig(pdf, es.objgen)
        if attack == "A3_hide_traces" or tgt is None:
            es.write(new)
        else:  # A5c_combined
            body, mimic_score = mimic(new, tgt)
            es.write(body, filter=pikepdf.Name.FlateDecode)
        if attack == "A5c_combined":
            n_decoys = add_decoys(pdf, dspec["page"], rng, compress=(lambda b: mimic(b, tgt)[0]) if tgt else None)
        for k, v in info_b.items():
            pdf.docinfo[pikepdf.Name(k)] = pikepdf.String(v)
        if ids_b:
            pdf.trailer[pikepdf.Name.ID] = pikepdf.Array([pikepdf.String(b) for b in ids_b])
        pdf.save(out)
    with open(out, "rb") as fh:
        raw = fh.read()
    with open(out, "wb") as fh:
        fh.write(strip_tool_words(raw))
    return out, n_decoys, mimic_score, tgt is not None

def build_missing(job):
    name, spec, attacks = job
    fitz.TOOLS.mupdf_display_errors(False)
    src = os.path.join(SRC, name)
    dspec = spec["dspec"]
    info_b, ids_b = read_meta_plain(src)
    with pikepdf.open(src) as p0:
        n_streams0 = len(page_streams(p0, dspec["page"]))
    with fitz.open(src) as d0:
        text0 = d0[dspec["page"]].get_text()
    rows, errs = [], []
    for attack in attacks:
        rng = random.Random(zlib.crc32((name + attack).encode()))
        try:
            out, n_dec, ms, has_tgt = build_one_fixed(src, dspec, attack, rng, info_b, ids_b)
            with fitz.open(out) as d:
                visible = int(d[dspec["page"]].get_text() != text0)
            with pikepdf.open(out) as p1:
                n_streams1 = len(page_streams(p1, dspec["page"]))
            f = extract_features(out)
            f.update({"doc": name, "attack": attack, "family": spec["family"], "split": spec["split"], "tool": spec["tool"],
                      "edit_visible": visible, "decoys_planned": n_dec, "decoys_found": n_streams1 - n_streams0,
                      "mimic_score": ms, "has_siblings": int(has_tgt)})
            rows.append(f)
        except Exception as ex:
            errs.append({"doc": name, "attack": attack, "reason": f"{type(ex).__name__}: {str(ex)[:60]}"})
        finally:
            p = os.path.join(ATK_PDF, f"{name[:-4]}__{attack}.pdf")
            if os.path.exists(p):
                os.remove(p)
    return rows, errs

# ---------- part 1: build only the missing (doc, attack) pairs ----------
feat = pd.read_csv(ATK_FEAT)
feat = feat.drop_duplicates(["doc", "attack"], keep="last")
have = set(zip(feat.doc, feat.attack))
jobs = []
for d, s in SPECS.items():
    miss = [a for a in ["A3_hide_traces", "A5c_combined"] if (d, a) not in have]
    if miss:
        jobs.append((d, s, miss))
print("Documents needing A3/A5c files:", len(jobs))

rows, fails = feat.to_dict("records"), []
with ProcessPool(max_workers=WORKERS) as pool_ex:
    futs = {pool_ex.schedule(build_missing, args=(j,), timeout=TIME_LIMIT): j[0] for j in jobs}
    for i, fu in enumerate(tqdm(as_completed(futs), total=len(futs), desc="Building A3 + A5c")):
        try:
            r, e = fu.result()
            rows += r
            fails += e
        except TimeoutError:
            fails.append({"doc": futs[fu], "attack": "A3/A5c", "reason": "Timeout"})
        except Exception as ex:
            fails.append({"doc": futs[fu], "attack": "A3/A5c", "reason": f"{type(ex).__name__}: {str(ex)[:60]}"})
        if (i + 1) % 200 == 0:
            pd.DataFrame(rows).to_csv(ATK_FEAT, index=False)

feat = pd.DataFrame(rows).drop_duplicates(["doc", "attack"], keep="last")
feat.to_csv(ATK_FEAT, index=False)
print("\nFailures:", len(fails), pd.Series([x["reason"].split(":")[0] for x in fails]).value_counts().to_dict() if fails else "")
print("\nFiles per attack:")
print(feat.attack.value_counts().reindex(ATTACKS).to_string())
print("\n=== CHECKS PER ATTACK ===")
chk = feat.groupby("attack").agg(edit_visible_pct=("edit_visible", lambda s: round(s.mean() * 100, 1)),
                                 decoys_found=("decoys_found", "mean"),
                                 has_producer_pct=("has_producer", lambda s: round(s.mean() * 100, 1)),
                                 tool_word_pct=("tool_word_in_file", lambda s: round(s.mean() * 100, 1)))
print(chk.reindex(ATTACKS).round(2).to_string())

# ---------- part 2: which of the 6 features does the A2b mimic fail to match? ----------
FEATS6 = ["hdr", "zlevel", "sync_end", "adler_ok", "tail_junk", "ratio"]

def diag(job):
    name, spec = job
    src = os.path.join(SRC, name)
    dspec = spec["dspec"]
    with pikepdf.open(src) as pdf:
        es = page_streams(pdf, dspec["page"])[dspec["k"]]
        data = es.read_bytes()
        new = data[:dspec["start"]] + dspec["new"] + data[dspec["end"]:]
        tgt = target_sig(pdf, es.objgen)
    if tgt is None:
        return None
    body, _ = mimic(new, tgt)
    s = sig_of(body)
    m = {f: int(s[f] == tgt[f]) for f in FEATS6[:-1]}
    m["ratio"] = int(abs(s["ratio"] - tgt["ratio"]) < 0.15)
    return {"family": spec["family"], **{f"match_{k}": v for k, v in m.items()},
            **{f"tgt_{k}": tgt[k] for k in ["zlevel", "sync_end", "adler_ok", "tail_junk"]},
            "tgt_ratio": tgt["ratio"], "mimic_ratio": s["ratio"]}

sample = random.Random(1).sample(list(SPECS.items()), 300)
drows = []
with ProcessPool(max_workers=WORKERS) as pool_ex:
    futs = [pool_ex.schedule(diag, args=(j,), timeout=TIME_LIMIT) for j in sample]
    for fu in tqdm(as_completed(futs), total=len(futs), desc="Diagnosing A2b"):
        try:
            r = fu.result()
            if r:
                drows.append(r)
        except Exception:
            pass
dg = pd.DataFrame(drows)
print("\n=== A2b DIAGNOSIS: % of documents where the mimic MATCHES each feature ===")
print(dg.groupby("family")[[f"match_{f}" for f in FEATS6]].mean().mul(100).round(1).to_string())
print("\n=== What the original pages look like (target habits, % of documents) ===")
print(dg.groupby("family").agg(reproducible=("tgt_zlevel", lambda s: round((s > 0).mean() * 100, 1)),
                               sync_end=("tgt_sync_end", lambda s: round(s.mean() * 100, 1)),
                               adler_ok=("tgt_adler_ok", lambda s: round(s.mean() * 100, 1)),
                               tail_junk=("tgt_tail_junk", lambda s: round(s.mean() * 100, 1)),
                               target_ratio=("tgt_ratio", "median"),
                               mimic_ratio=("mimic_ratio", "median")).round(3).to_string())
dg.to_csv(f"{ATK}/a2b_diagnosis.csv", index=False)
print(f"\nSaved: {ATK_FEAT} and {ATK}/a2b_diagnosis.csv")

Documents needing A3/A5c files: 994


Building A3 + A5c:   0%|          | 0/994 [00:00<?, ?it/s]


Failures: 0 

Files per attack:
attack
A0_plain            1003
A2a_mimic_simple    1003
A2b_mimic_aware     1003
A3_hide_traces      1003
A4_decoys           1003
A5c_combined        1003

=== CHECKS PER ATTACK ===
                  edit_visible_pct  decoys_found  has_producer_pct  tool_word_pct
attack                                                                           
A0_plain                      99.6          0.00             100.0            0.0
A2a_mimic_simple              99.6          0.00             100.0            0.0
A2b_mimic_aware               99.6          0.00             100.0            0.0
A3_hide_traces                99.6          0.00             100.0            0.0
A4_decoys                     99.6          3.01             100.0            0.0
A5c_combined                  99.6          3.00             100.0            0.0


Diagnosing A2b:   0%|          | 0/300 [00:00<?, ?it/s]


=== A2b DIAGNOSIS: % of documents where the mimic MATCHES each feature ===
                           match_hdr  match_zlevel  match_sync_end  match_adler_ok  match_tail_junk  match_ratio
family                                                                                                          
acrobat distiller for            1.8         100.0           100.0           100.0            100.0         89.5
acrobat distiller windows        0.7          99.3           100.0           100.0            100.0         94.8
acrobat pdfwriter for            0.0         100.0           100.0           100.0            100.0         83.3
adobe pdf library                0.0         100.0           100.0           100.0            100.0         83.3
corel pdf engine                 0.0         100.0           100.0           100.0            100.0        100.0

=== What the original pages look like (target habits, % of documents) ===
                           reproducible  sync_end  adler_o

In [7]:
# CELL 28c — Fix A2b mimic (copy window size + level from the original's zlib header), rebuild A2b + A5c, re-diagnose, check real headers and Corel reproducibility
import os, zlib, random, collections
import numpy as np
import pandas as pd
import pikepdf
import pymupdf as fitz
from pebble import ProcessPool
from concurrent.futures import TimeoutError, as_completed
from tqdm.auto import tqdm

def level_flag(level, strat):
    # exactly how zlib fills FLEVEL (2nd header byte, top 2 bits)
    if strat in (zlib.Z_HUFFMAN_ONLY, zlib.Z_RLE) or level < 2:
        return 0
    return 1 if level < 6 else (2 if level == 6 else 3)

def mimic(data, tgt):
    """Compress `data` so its measured signature matches tgt. Header: window size from CMF, level from FLG."""
    std = {lv: zlib.compress(data, lv) for lv in range(1, 10)}
    if tgt["zlevel"] > 0:                                   # original pages are reproducible by standard zlib
        cands = [std[tgt["zlevel"]]]
    else:
        cmf, flg = (tgt["hdr"] >> 8) & 0xFF, tgt["hdr"] & 0xFF
        if tgt["hdr"] < 0 or (cmf & 0x0F) != 8:
            cmf, flg = 0x78, 0x9C
        wbits = min(15, max(9, (cmf >> 4) + 8))
        flevel = flg >> 6
        cands = []
        for lv in range(0, 10):
            for st in STRATS:
                if level_flag(lv, st) != flevel:
                    continue
                for mem in (1, 3, 5, 7, 8, 9):
                    co = zlib.compressobj(lv, zlib.DEFLATED, wbits, mem, st)
                    if tgt["sync_end"]:
                        b = co.compress(data) + co.flush(zlib.Z_SYNC_FLUSH) + co.flush(zlib.Z_FINISH)
                    else:
                        b = co.compress(data) + co.flush()
                    cands.append(b)
        if not cands:
            cands = [std[6]]
    best, best_key = None, None
    for b in cands:
        if tgt["tail_junk"]:
            b = b + b"\n"
        s = sig_of(b, std)
        score = sum([s["hdr"] == tgt["hdr"], s["zlevel"] == tgt["zlevel"], s["sync_end"] == tgt["sync_end"],
                     s["adler_ok"] == tgt["adler_ok"], s["tail_junk"] == tgt["tail_junk"],
                     abs(s["ratio"] - tgt["ratio"]) < 0.15])
        key = (score, -abs(s["ratio"] - tgt["ratio"]))
        if best_key is None or key > best_key:
            best, best_key = b, key
    return best, best_key[0]

# ---------- part 1: rebuild only A2b and A5c with the fixed mimic ----------
REBUILD = ["A2b_mimic_aware", "A5c_combined"]

def rebuild(job):
    name, spec = job
    fitz.TOOLS.mupdf_display_errors(False)
    src = os.path.join(SRC, name)
    dspec = spec["dspec"]
    info_b, ids_b = read_meta_plain(src)
    with pikepdf.open(src) as p0:
        n_streams0 = len(page_streams(p0, dspec["page"]))
        tgt = target_sig(p0, page_streams(p0, dspec["page"])[dspec["k"]].objgen)
    with fitz.open(src) as d0:
        text0 = d0[dspec["page"]].get_text()
    rows, errs = [], []
    for attack in REBUILD:
        rng = random.Random(zlib.crc32((name + attack).encode()))
        try:
            if attack == "A2b_mimic_aware":
                out, n_dec, ms, has_tgt = build_one(src, dspec, attack, rng, {}, None)
            else:
                out, n_dec, ms, has_tgt = build_one_fixed(src, dspec, attack, rng, info_b, ids_b)
            with fitz.open(out) as d:
                visible = int(d[dspec["page"]].get_text() != text0)
            with pikepdf.open(out) as p1:
                n_streams1 = len(page_streams(p1, dspec["page"]))
                es1 = page_streams(p1, dspec["page"])[dspec["k"]]
                raw1 = es1.read_raw_bytes()
                hdr_ok = int(tgt is not None and len(raw1) >= 2 and raw1[0] * 256 + raw1[1] == tgt["hdr"])
            f = extract_features(out)
            f.update({"doc": name, "attack": attack, "family": spec["family"], "split": spec["split"], "tool": spec["tool"],
                      "edit_visible": visible, "decoys_planned": n_dec, "decoys_found": n_streams1 - n_streams0,
                      "mimic_score": ms, "has_siblings": int(has_tgt), "hdr_matched": hdr_ok})
            rows.append(f)
        except Exception as ex:
            errs.append({"doc": name, "attack": attack, "reason": f"{type(ex).__name__}: {str(ex)[:60]}"})
        finally:
            p = os.path.join(ATK_PDF, f"{name[:-4]}__{attack}.pdf")
            if os.path.exists(p):
                os.remove(p)
    return rows, errs

new_rows, fails = [], []
with ProcessPool(max_workers=WORKERS) as pool_ex:
    futs = {pool_ex.schedule(rebuild, args=(j,), timeout=TIME_LIMIT): j[0] for j in SPECS.items()}
    for fu in tqdm(as_completed(futs), total=len(futs), desc="Rebuilding A2b + A5c"):
        try:
            r, e = fu.result()
            new_rows += r
            fails += e
        except TimeoutError:
            fails.append({"doc": futs[fu], "attack": "A2b/A5c", "reason": "Timeout"})
        except Exception as ex:
            fails.append({"doc": futs[fu], "attack": "A2b/A5c", "reason": f"{type(ex).__name__}: {str(ex)[:60]}"})

feat = pd.read_csv(ATK_FEAT)
feat = feat[~feat.attack.isin(REBUILD)]
feat = pd.concat([feat, pd.DataFrame(new_rows)], ignore_index=True).drop_duplicates(["doc", "attack"], keep="last")
feat.to_csv(ATK_FEAT, index=False)

nr = pd.DataFrame(new_rows)
print("\nRebuilt files:", len(nr), "| failures:", len(fails),
      pd.Series([x["reason"].split(":")[0] for x in fails]).value_counts().to_dict() if fails else "")
print("Files per attack now:", feat.attack.value_counts().reindex(ATTACKS).to_dict())
sib = nr[nr.has_siblings == 1]
print("\n=== FIXED MIMIC (documents with sibling streams) ===")
print(sib.groupby(["attack", "family"]).agg(header_matched_pct=("hdr_matched", lambda s: round(s.mean() * 100, 1)),
                                           all6_matched_pct=("mimic_score", lambda s: round((s == 6).mean() * 100, 1)),
                                           mean_matched=("mimic_score", "mean"),
                                           edit_visible_pct=("edit_visible", lambda s: round(s.mean() * 100, 1))).round(2).to_string())

# ---------- part 2: re-diagnose which features the fixed mimic matches (same 300 documents) ----------
drows = []
with ProcessPool(max_workers=WORKERS) as pool_ex:
    futs = [pool_ex.schedule(diag, args=(j,), timeout=TIME_LIMIT) for j in sample]
    for fu in tqdm(as_completed(futs), total=len(futs), desc="Re-diagnosing A2b"):
        try:
            r = fu.result()
            if r:
                drows.append(r)
        except Exception:
            pass
dg = pd.DataFrame(drows)
print("\n=== A2b DIAGNOSIS AFTER FIX: % of documents where the mimic MATCHES each feature ===")
print(dg.groupby("family")[[f"match_{f}" for f in FEATS6]].mean().mul(100).round(1).to_string())

# ---------- part 3: what the original page headers really are, and real reproducibility per family ----------
def hdr_check(job):
    name, spec = job
    out = []
    with pikepdf.open(os.path.join(SRC, name)) as pdf:
        for i in range(len(pdf.pages)):
            for s in page_streams(pdf, i):
                flt = s.get("/Filter")
                fl = str(flt) if flt is not None else "none"
                if fl != "/FlateDecode":
                    out.append({"family": spec["family"], "filter": fl, "hdr": None, "repro": None})
                    continue
                raw = s.read_raw_bytes()
                try:
                    dob = zlib.decompressobj()
                    dec = dob.decompress(raw)
                    body = raw[:len(raw) - len(dob.unused_data)]
                    rep = int(len(dec) <= 2_000_000 and any(zlib.compress(dec, lv) == body for lv in range(1, 10)))
                except Exception:
                    rep = 0
                out.append({"family": spec["family"], "filter": fl, "hdr": f"{raw[0]:02X} {raw[1]:02X}" if len(raw) > 1 else "?", "repro": rep})
    return out

hrows = []
with ProcessPool(max_workers=WORKERS) as pool_ex:
    futs = [pool_ex.schedule(hdr_check, args=(j,), timeout=TIME_LIMIT) for j in sample]
    for fu in tqdm(as_completed(futs), total=len(futs), desc="Checking original headers"):
        try:
            hrows += fu.result()
        except Exception:
            pass
hd = pd.DataFrame(hrows)
print("\n=== ORIGINAL PAGE STREAMS: filter used, per family (% of streams) ===")
print(pd.crosstab(hd.family, hd["filter"], normalize="index").mul(100).round(1).to_string())
fl = hd[hd["filter"] == "/FlateDecode"]
print("\n=== ORIGINAL FLATE PAGE STREAMS: reproducible by standard zlib (% of streams) ===")
print(fl.groupby("family").repro.mean().mul(100).round(1).to_string())
print("\n=== MOST COMMON ZLIB HEADERS per family (% of Flate page streams) ===")
for fam, g in fl.groupby("family"):
    top = g.hdr.value_counts(normalize=True).mul(100).round(1).head(3).to_dict()
    print(f"{fam:28s}: {top}")
dg.to_csv(f"{ATK}/a2b_diagnosis_fixed.csv", index=False)
print(f"\nSaved: {ATK_FEAT} and {ATK}/a2b_diagnosis_fixed.csv")

Rebuilding A2b + A5c:   0%|          | 0/1003 [00:00<?, ?it/s]


Rebuilt files: 2006 | failures: 0 
Files per attack now: {'A0_plain': 1003, 'A2a_mimic_simple': 1003, 'A2b_mimic_aware': 1003, 'A3_hide_traces': 1003, 'A4_decoys': 1003, 'A5c_combined': 1003}

=== FIXED MIMIC (documents with sibling streams) ===
                                           header_matched_pct  all6_matched_pct  mean_matched  edit_visible_pct
attack          family                                                                                         
A2b_mimic_aware acrobat distiller for                   100.0              93.0          5.93              98.9
                acrobat distiller windows               100.0              95.0          5.95              99.6
                acrobat pdfwriter for                   100.0              81.1          5.81             100.0
                adobe pdf library                       100.0              89.7          5.90             100.0
                corel pdf engine                        100.0             100.0  

Re-diagnosing A2b:   0%|          | 0/300 [00:00<?, ?it/s]


=== A2b DIAGNOSIS AFTER FIX: % of documents where the mimic MATCHES each feature ===
                           match_hdr  match_zlevel  match_sync_end  match_adler_ok  match_tail_junk  match_ratio
family                                                                                                          
acrobat distiller for          100.0         100.0           100.0           100.0            100.0         91.2
acrobat distiller windows      100.0         100.0           100.0           100.0            100.0         95.4
acrobat pdfwriter for          100.0         100.0           100.0           100.0            100.0         83.3
adobe pdf library              100.0         100.0           100.0           100.0            100.0         83.3
corel pdf engine               100.0         100.0           100.0           100.0            100.0        100.0


Checking original headers:   0%|          | 0/300 [00:00<?, ?it/s]


=== ORIGINAL PAGE STREAMS: filter used, per family (% of streams) ===
filter                     /FlateDecode  /LZWDecode  none  pikepdf.Array([ "/ASCII85Decode" ])  pikepdf.Array([ "/ASCII85Decode", "/FlateDecode" ])  pikepdf.Array([ "/FlateDecode" ])
family                                                                                                                                                                                
acrobat distiller for              81.8        17.2   1.0                                  0.0                                                  0.0                                0.0
acrobat distiller windows          99.9         0.0   0.0                                  0.0                                                  0.1                                0.0
acrobat pdfwriter for              84.7        15.3   0.0                                  0.0                                                  0.0                                0.0
adobe pdf libr

In [8]:
# CELL 28d — Treat [/FlateDecode] arrays as Flate (Corel), rebuild A2a + A2b + A5c, re-diagnose, and test if Adobe pages are reproducible by ANY zlib setting with their window size
import os, zlib, random, collections
import numpy as np
import pandas as pd
import pikepdf
import pymupdf as fitz
from pebble import ProcessPool
from concurrent.futures import TimeoutError, as_completed
from tqdm.auto import tqdm

def filter_names(s):
    flt = s.get("/Filter")
    if isinstance(flt, pikepdf.Array):
        return [str(x) for x in flt]
    return [] if flt is None else [str(flt)]

def target_sig(pdf, skip):
    sigs = []
    for i in range(len(pdf.pages)):
        for s in page_streams(pdf, i):
            if s.objgen == skip or filter_names(s) != ["/FlateDecode"]:
                continue
            try:
                sigs.append(sig_of(s.read_raw_bytes()))
            except Exception:
                pass
    if not sigs:
        return None
    key = lambda d: (d["hdr"], d["zlevel"], d["sync_end"], d["adler_ok"], d["tail_junk"])
    best = collections.Counter(key(d) for d in sigs).most_common(1)[0][0]
    return dict(zip(["hdr", "zlevel", "sync_end", "adler_ok", "tail_junk"], best),
                ratio=float(np.median([d["ratio"] for d in sigs])))

# ---------- part 1: rebuild every attack that uses the target ----------
REBUILD = ["A2a_mimic_simple", "A2b_mimic_aware", "A5c_combined"]

def rebuild2(job):
    name, spec = job
    fitz.TOOLS.mupdf_display_errors(False)
    src = os.path.join(SRC, name)
    dspec = spec["dspec"]
    info_b, ids_b = read_meta_plain(src)
    with pikepdf.open(src) as p0:
        n_streams0 = len(page_streams(p0, dspec["page"]))
        tgt = target_sig(p0, page_streams(p0, dspec["page"])[dspec["k"]].objgen)
    with fitz.open(src) as d0:
        text0 = d0[dspec["page"]].get_text()
    rows, errs = [], []
    for attack in REBUILD:
        rng = random.Random(zlib.crc32((name + attack).encode()))
        try:
            if attack == "A5c_combined":
                out, n_dec, ms, has_tgt = build_one_fixed(src, dspec, attack, rng, info_b, ids_b)
            else:
                out, n_dec, ms, has_tgt = build_one(src, dspec, attack, rng, {}, None)
            with fitz.open(out) as d:
                visible = int(d[dspec["page"]].get_text() != text0)
            with pikepdf.open(out) as p1:
                n_streams1 = len(page_streams(p1, dspec["page"]))
                raw1 = page_streams(p1, dspec["page"])[dspec["k"]].read_raw_bytes()
                hdr_ok = int(tgt is not None and len(raw1) >= 2 and raw1[0] * 256 + raw1[1] == tgt["hdr"])
            f = extract_features(out)
            f.update({"doc": name, "attack": attack, "family": spec["family"], "split": spec["split"], "tool": spec["tool"],
                      "edit_visible": visible, "decoys_planned": n_dec, "decoys_found": n_streams1 - n_streams0,
                      "mimic_score": ms, "has_siblings": int(has_tgt), "hdr_matched": hdr_ok,
                      "tgt_reproducible": int(tgt is not None and tgt["zlevel"] > 0)})
            rows.append(f)
        except Exception as ex:
            errs.append({"doc": name, "attack": attack, "reason": f"{type(ex).__name__}: {str(ex)[:60]}"})
        finally:
            p = os.path.join(ATK_PDF, f"{name[:-4]}__{attack}.pdf")
            if os.path.exists(p):
                os.remove(p)
    return rows, errs

new_rows, fails = [], []
with ProcessPool(max_workers=WORKERS) as pool_ex:
    futs = {pool_ex.schedule(rebuild2, args=(j,), timeout=TIME_LIMIT): j[0] for j in SPECS.items()}
    for fu in tqdm(as_completed(futs), total=len(futs), desc="Rebuilding A2a + A2b + A5c"):
        try:
            r, e = fu.result()
            new_rows += r
            fails += e
        except TimeoutError:
            fails.append({"doc": futs[fu], "attack": "rebuild", "reason": "Timeout"})
        except Exception as ex:
            fails.append({"doc": futs[fu], "attack": "rebuild", "reason": f"{type(ex).__name__}: {str(ex)[:60]}"})

feat = pd.read_csv(ATK_FEAT)
feat = feat[~feat.attack.isin(REBUILD)]
feat = pd.concat([feat, pd.DataFrame(new_rows)], ignore_index=True).drop_duplicates(["doc", "attack"], keep="last")
feat.to_csv(ATK_FEAT, index=False)

nr = pd.DataFrame(new_rows)
print("\nRebuilt files:", len(nr), "| failures:", len(fails),
      pd.Series([x["reason"].split(":")[0] for x in fails]).value_counts().to_dict() if fails else "")
print("Files per attack now:", feat.attack.value_counts().reindex(ATTACKS).to_dict())
sib = nr[nr.has_siblings == 1]
print("\n=== MIMIC ATTACKS AFTER FIX (documents with sibling streams) ===")
print(sib.groupby(["attack", "family"]).agg(target_reproducible_pct=("tgt_reproducible", lambda s: round(s.mean() * 100, 1)),
                                           header_matched_pct=("hdr_matched", lambda s: round(s.mean() * 100, 1)),
                                           all6_matched_pct=("mimic_score", lambda s: round((s == 6).mean() * 100, 1)),
                                           edit_visible_pct=("edit_visible", lambda s: round(s.mean() * 100, 1))).round(1).to_string())

# ---------- part 2: re-diagnose (same 300 documents) ----------
drows = []
with ProcessPool(max_workers=WORKERS) as pool_ex:
    futs = [pool_ex.schedule(diag, args=(j,), timeout=TIME_LIMIT) for j in sample]
    for fu in tqdm(as_completed(futs), total=len(futs), desc="Re-diagnosing A2b"):
        try:
            r = fu.result()
            if r:
                drows.append(r)
        except Exception:
            pass
dg = pd.DataFrame(drows)
print("\n=== A2b DIAGNOSIS: % of documents where the mimic MATCHES each feature ===")
print(dg.groupby("family")[[f"match_{f}" for f in FEATS6]].mean().mul(100).round(1).to_string())

# ---------- part 3: can ANY zlib setting with the header's window size reproduce the original pages byte-for-byte? ----------
def deep_repro(job):
    name, spec = job
    out = []
    with pikepdf.open(os.path.join(SRC, name)) as pdf:
        n = 0
        for i in range(len(pdf.pages)):
            for s in page_streams(pdf, i):
                if n >= 6:
                    break
                if filter_names(s) != ["/FlateDecode"]:
                    continue
                raw = s.read_raw_bytes()
                if len(raw) < 6 or len(raw) > 120_000:
                    continue
                try:
                    dob = zlib.decompressobj()
                    dec = dob.decompress(raw)
                    body = raw[:len(raw) - len(dob.unused_data)]
                except Exception:
                    continue
                wb = min(15, max(9, (raw[0] >> 4) + 8))
                hit = None
                for lv in range(0, 10):
                    for st in STRATS:
                        for mem in range(1, 10):
                            co = zlib.compressobj(lv, zlib.DEFLATED, wb, mem, st)
                            if co.compress(dec) + co.flush() == body:
                                hit = (lv, st, mem)
                                break
                        if hit:
                            break
                    if hit:
                        break
                out.append({"family": spec["family"], "hdr": f"{raw[0]:02X} {raw[1]:02X}", "window_bits": wb,
                            "reproducible_any_zlib": int(hit is not None), "setting": str(hit)})
                n += 1
    return out

rrows = []
with ProcessPool(max_workers=WORKERS) as pool_ex:
    futs = [pool_ex.schedule(deep_repro, args=(j,), timeout=TIME_LIMIT) for j in sample[:120]]
    for fu in tqdm(as_completed(futs), total=len(futs), desc="Deep reproducibility test"):
        try:
            rrows += fu.result()
        except Exception:
            pass
rr = pd.DataFrame(rrows)
print("\n=== DEEP TEST: original Flate page streams reproducible by ANY zlib setting with their own window size ===")
print(rr.groupby(["family", "hdr"]).agg(streams=("reproducible_any_zlib", "size"),
                                        reproducible_pct=("reproducible_any_zlib", lambda s: round(s.mean() * 100, 1))).to_string())
if rr.reproducible_any_zlib.any():
    print("\nSettings that reproduced them (level, strategy, memLevel):")
    print(rr[rr.reproducible_any_zlib == 1].setting.value_counts().head(5).to_string())
rr.to_csv(f"{ATK}/deep_repro.csv", index=False)
print(f"\nSaved: {ATK_FEAT} and {ATK}/deep_repro.csv")

Rebuilding A2a + A2b + A5c:   0%|          | 0/1003 [00:00<?, ?it/s]


Rebuilt files: 3009 | failures: 0 
Files per attack now: {'A0_plain': 1003, 'A2a_mimic_simple': 1003, 'A2b_mimic_aware': 1003, 'A3_hide_traces': 1003, 'A4_decoys': 1003, 'A5c_combined': 1003}

=== MIMIC ATTACKS AFTER FIX (documents with sibling streams) ===
                                            target_reproducible_pct  header_matched_pct  all6_matched_pct  edit_visible_pct
attack           family                                                                                                    
A2a_mimic_simple acrobat distiller for                          1.6                 1.6               0.0              98.9
                 acrobat distiller windows                      0.6                 0.6               0.0              99.6
                 acrobat pdfwriter for                          0.0                 0.0               0.0             100.0
                 adobe pdf library                              0.0                 0.0               0.0             100

Re-diagnosing A2b:   0%|          | 0/300 [00:00<?, ?it/s]


=== A2b DIAGNOSIS: % of documents where the mimic MATCHES each feature ===
                           match_hdr  match_zlevel  match_sync_end  match_adler_ok  match_tail_junk  match_ratio
family                                                                                                          
acrobat distiller for          100.0         100.0           100.0           100.0            100.0         91.2
acrobat distiller windows      100.0         100.0           100.0           100.0            100.0         95.4
acrobat pdfwriter for          100.0         100.0           100.0           100.0            100.0         83.3
adobe pdf library              100.0         100.0           100.0           100.0            100.0         83.3
corel pdf engine               100.0          81.2           100.0           100.0            100.0        100.0


Deep reproducibility test:   0%|          | 0/120 [00:00<?, ?it/s]


=== DEEP TEST: original Flate page streams reproducible by ANY zlib setting with their own window size ===
                                 streams  reproducible_pct
family                    hdr                             
acrobat distiller for     48 89      131               8.4
acrobat distiller windows 48 89      363               3.9
                          68 DE        5               0.0
acrobat pdfwriter for     48 89       88              86.4
adobe pdf library         48 89       18               0.0
corel pdf engine          78 DA       26             100.0

Settings that reproduced them (level, strategy, memLevel):
setting
(6, 0, 5)    83
(8, 0, 5)    14
(6, 0, 1)    10
(9, 0, 5)     6
(6, 0, 4)     5

Saved: /kaggle/working/attack/attack_features_v2.csv and /kaggle/working/attack/deep_repro.csv


In [9]:
# CELL 29 — Score all attacks with the UNCHANGED LightGBM detector (no defense): success rates, AUC vs honest re-saves, per family, rule localization
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupKFold

META = ["has_producer", "producer_len", "moddate_eq_creation", "has_xmp", "tool_word_in_file"]
L1 = ["size_log", "version", "binary_comment", "n_eof", "n_startxref", "xref_table", "xref_stream", "obj_streams",
      "linearized", "true_revisions", "eol_crlf", "eol_cr", "eol_lf", "n_obj_defs", "obj_order_increasing",
      "obj_gen_nonzero", "obj_max_over_count", "obj_redefined", "obj_newline_dict", "dict_open_space",
      "dict_close_space", "stream_kw_crlf", "endobj_newline_before", "reals_dot_zero", "has_ID", "tool_word_in_file", "n_pages"]
L23 = ["cs_n", "cs_frac_flate", "cs_frac_lzw", "cs_frac_none", "cs_n_filters", "cs_frac_zlib_repro", "cs_n_hdr",
       "cs_n_zlevel", "cs_frac_sync", "cs_frac_adler", "cs_frac_junk", "cs_frac_len_indirect", "cs_ratio_std",
       "cs_ratio_max_z", "cs_n_sig", "cs_majority_share", "cs_n_singletons"]
L4 = ["upd_n_objs", "upd_touch_content", "upd_has_annot", "upd_has_page", "upd_has_sig"]
L5 = ["l5_n_annots", "l5_n_freetext", "l5_n_text_notes", "l5_n_img_xobj", "l5_n_fonts", "l5_n_std14_unembedded",
      "l5_frac_pages_multi_stream", "l5_max_streams_per_page"]
FULL = sorted(set(META + L1 + L23 + L4 + L5))
PARAMS = dict(n_estimators=300, learning_rate=0.05, num_leaves=15, min_child_samples=10, subsample=0.8,
              subsample_freq=1, colsample_bytree=0.8, random_state=42, verbose=-1)

def fit(cols, train):
    return lgb.LGBMClassifier(**PARAMS).fit(train[cols], train["label"])

def auc(y, s):
    try:
        return round(roc_auc_score(y, s), 3)
    except ValueError:
        return np.nan

# ---------- the unchanged detector + its train-only threshold (same as CELL 23) ----------
tr = feat_full[feat_full.split == "train"].reset_index(drop=True)
MODEL = fit(FULL, tr)
oof = np.zeros(len(tr))
for a, b in GroupKFold(n_splits=5).split(tr, tr.label, tr.orig_file):
    oof[b] = fit(FULL, tr.iloc[a]).predict_proba(tr.iloc[b][FULL])[:, 1]
THR = float(np.quantile(oof[tr.label.values == 0], 0.95))
print(f"Detector: LightGBM FULL (L1-L5), trained on {tr.orig_file.nunique()} train docs | threshold {THR:.3f}")

# ---------- attacked files (dev only) and honest references ----------
atk = pd.read_csv(ATK_FEAT)
atk["edit_page"] = atk.doc.map(lambda d: SPECS[d]["dspec"]["page"])
g0_cs = feat_full[feat_full.variant == "G0_original"].set_index("orig_file").cs_n
atk["siblings"] = (atk.doc.map(g0_cs).fillna(0) >= 2).astype(int)
atk["score"] = MODEL.predict_proba(atk[FULL])[:, 1]
atk["detected"] = (atk.score > THR).astype(int)
dev = atk[atk.split == "dev"].copy()

devf = feat_full[feat_full.split == "dev"].copy()
devf["score"] = MODEL.predict_proba(devf[FULL])[:, 1]
g1 = devf[devf.variant == "G1_resave"]
f1 = devf[devf.variant == "F1_digit_resave"]
pk_docs = {d for d, s in SPECS.items() if s["tool"] == "pikepdf" and s["split"] == "dev"}
print(f"Dev documents attacked: {dev.doc.nunique()} | with sibling page-streams: {dev[dev.siblings == 1].doc.nunique()}"
      f" | docs whose honest re-save was made by pikepdf: {len(pk_docs)}")
print(f"Reference: original F1 forgery (tool-native) detected {f1.score.gt(THR).mean() * 100:.1f}% | "
      f"honest re-saves G1 falsely flagged {g1.score.gt(THR).mean() * 100:.1f}%")

# ---------- table 1: per attack ----------
rows = []
for a in ATTACKS:
    s = dev[dev.attack == a]
    s_sib = s[s.siblings == 1]
    y_all = np.r_[np.ones(len(s)), np.zeros(len(g1))]
    sc_all = np.r_[s.score.values, g1.score.values]
    s_pk = s[s.doc.isin(pk_docs)]
    g1_pk = g1[g1.orig_file.isin(pk_docs)]
    rows.append({"Attack": a, "n": len(s),
                 "Detected %": round(s.detected.mean() * 100, 1),
                 "Fools detector %": round((1 - s.detected.mean()) * 100, 1),
                 "Fools % (docs w/ siblings)": round((1 - s_sib.detected.mean()) * 100, 1),
                 "AUC vs honest re-save (all)": auc(y_all, sc_all),
                 "AUC vs honest pikepdf re-save": auc(np.r_[np.ones(len(s_pk)), np.zeros(len(g1_pk))],
                                                      np.r_[s_pk.score.values, g1_pk.score.values])})
print("\n=== TABLE 1: ATTACKS vs UNCHANGED DETECTOR (dev set) ===")
print(pd.DataFrame(rows).to_string(index=False))

# ---------- table 2: per family (docs with siblings) ----------
print("\n=== TABLE 2: % OF ATTACKED FAKES THAT FOOL THE DETECTOR, per family (dev, docs with sibling streams) ===")
fam = dev[dev.siblings == 1].groupby(["family", "attack"]).detected.mean().unstack()[ATTACKS]
print((100 - fam * 100).round(1).to_string())
print("Docs per family:", dev[dev.siblings == 1].groupby("family").doc.nunique().to_dict())

# ---------- table 3: rule localization under attack ----------
print("\n=== TABLE 3: RULE PAGE-FINDER UNDER ATTACK (dev) ===")
lr = []
for a in ATTACKS:
    s = dev[dev.attack == a]
    lp = s.loc_page.fillna(-1)
    fired = lp >= 0
    lr.append({"Attack": a, "Rule fires %": round(fired.mean() * 100, 1),
               "Correct when fires %": round((lp[fired] == s.edit_page[fired]).mean() * 100, 1) if fired.any() else np.nan,
               "Correct overall %": round((lp == s.edit_page).mean() * 100, 1)})
print(pd.DataFrame(lr).to_string(index=False))

atk.to_csv(f"{ATK}/attack_scores_nodefense.csv", index=False)
print(f"\nSaved: {ATK}/attack_scores_nodefense.csv")

Detector: LightGBM FULL (L1-L5), trained on 2319 train docs | threshold 0.531
Dev documents attacked: 403 | with sibling page-streams: 389 | docs whose honest re-save was made by pikepdf: 129
Reference: original F1 forgery (tool-native) detected 99.5% | honest re-saves G1 falsely flagged 5.0%

=== TABLE 1: ATTACKS vs UNCHANGED DETECTOR (dev set) ===
          Attack   n  Detected %  Fools detector %  Fools % (docs w/ siblings)  AUC vs honest re-save (all)  AUC vs honest pikepdf re-save
        A0_plain 403        99.5               0.5                         0.5                        0.994                          0.993
A2a_mimic_simple 403        99.5               0.5                         0.5                        0.995                          0.995
 A2b_mimic_aware 403        15.4              84.6                        87.7                        0.344                          0.529
  A3_hide_traces 403        99.5               0.5                         0.5              

In [10]:
# CELL 30 — Defenses: D1 adversarial training, D2 robust features, D3 monotonic constraints, D1+D2+D3 — re-score attacks, check clean accuracy and false alarms
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupKFold
from tqdm.auto import tqdm

EASY_FAKE = META + ["has_ID"]                                   # things a forger can set freely
ROBUST = [c for c in FULL if c not in EASY_FAKE]
UP = ["cs_n_hdr", "cs_n_zlevel", "cs_n_sig", "cs_n_singletons", "cs_ratio_std", "cs_ratio_max_z", "cs_n_filters"]
DOWN = ["cs_majority_share"]
def mono(cols):
    return [1 if c in UP else (-1 if c in DOWN else 0) for c in cols]

base_tr = feat_full[feat_full.split == "train"].copy()
adv_tr = atk[atk.split == "train"].copy()
adv_tr["label"], adv_tr["orig_file"] = 1, adv_tr["doc"]
aug_tr = pd.concat([base_tr[FULL + ["label", "orig_file"]], adv_tr[FULL + ["label", "orig_file"]]], ignore_index=True)
print(f"Training rows: normal {len(base_tr)} | with attacked train-doc fakes {len(aug_tr)} (+{len(adv_tr)} from {adv_tr.doc.nunique()} docs)")

NO_EV = ["F2_digit_recompress", "F7_stamp_fake", "F8_textbox_fake", "G2_recompress", "G5_stamp_honest", "G6_textbox_honest"]
EVm = ~devf.variant.isin(NO_EV).values
honest_dev = devf.label.values == 0
g1_pk = (devf.variant == "G1_resave").values & devf.orig_file.isin(pk_docs).values
f1m = (devf.variant == "F1_digit_resave").values
dev_atk = atk[atk.split == "dev"]

VARIANTS = [("No defense", FULL, base_tr, False),
            ("D1 adversarial training", FULL, aug_tr, False),
            ("D2 robust features", ROBUST, base_tr, False),
            ("D3 monotonic constraints", FULL, base_tr, True),
            ("D1+D2+D3", ROBUST, aug_tr, True)]

def fit_params(cols, monotone):
    p = dict(PARAMS)
    if monotone:
        p["monotone_constraints"] = mono(cols)
        p["monotone_constraints_method"] = "advanced"
    return p

results, per_family = [], {}
for name, cols, train, monotone in tqdm(VARIANTS, desc="Defenses"):
    p = fit_params(cols, monotone)
    train = train.reset_index(drop=True)
    oof = np.zeros(len(train))
    for a, b in GroupKFold(n_splits=5).split(train, train.label, train.orig_file):
        oof[b] = lgb.LGBMClassifier(**p).fit(train.iloc[a][cols], train.iloc[a].label).predict_proba(train.iloc[b][cols])[:, 1]
    thr = float(np.quantile(oof[train.label.values == 0], 0.95))
    m = lgb.LGBMClassifier(**p).fit(train[cols], train.label)
    sd = m.predict_proba(devf[cols])[:, 1]
    sa = m.predict_proba(dev_atk[cols])[:, 1]
    r = {"Defense": name, "Threshold": round(thr, 3),
         "Clean AUC (evidence)": round(roc_auc_score(devf.label.values[EVm], sd[EVm]), 4),
         "F1 caught %": round((sd[f1m] > thr).mean() * 100, 1),
         "False alarms, all honest %": round((sd[honest_dev] > thr).mean() * 100, 1),
         "False alarms, honest pikepdf re-save %": round((sd[g1_pk] > thr).mean() * 100, 1)}
    for a in ATTACKS:
        mk = (dev_atk.attack == a).values
        r[f"{a} caught %"] = round((sa[mk] > thr).mean() * 100, 1)
    results.append(r)
    det = pd.Series(sa > thr, index=dev_atk.index)
    per_family[name] = dev_atk[dev_atk.attack == "A2b_mimic_aware"].assign(d=det).groupby("family").d.mean().mul(100).round(1)

res = pd.DataFrame(results).set_index("Defense")
print("\n=== TABLE 4: DEFENSES — clean accuracy and false alarms (dev) ===")
print(res[["Threshold", "Clean AUC (evidence)", "F1 caught %", "False alarms, all honest %",
           "False alarms, honest pikepdf re-save %"]].to_string())
print("\n=== TABLE 5: DEFENSES — % OF EACH ATTACK CAUGHT (dev, 403 docs) ===")
print(res[[f"{a} caught %" for a in ATTACKS]].rename(columns=lambda c: c.replace(" caught %", "")).to_string())
print("\n=== TABLE 6: A2b (feature-aware mimic) CAUGHT %, per family ===")
print(pd.DataFrame(per_family).to_string())

res.to_csv(f"{ATK}/defenses_d1_d3.csv")
print(f"\nSaved: {ATK}/defenses_d1_d3.csv")

Training rows: normal 34785 | with attacked train-doc fakes 38385 (+3600 from 600 docs)


Defenses:   0%|          | 0/5 [00:00<?, ?it/s]


=== TABLE 4: DEFENSES — clean accuracy and false alarms (dev) ===
                          Threshold  Clean AUC (evidence)  F1 caught %  False alarms, all honest %  False alarms, honest pikepdf re-save %
Defense                                                                                                                                   
No defense                    0.531                0.9981         99.5                         4.1                                    11.6
D1 adversarial training       0.542                0.9973         99.3                         4.9                                    25.6
D2 robust features            0.529                0.9982         99.3                         3.9                                    11.6
D3 monotonic constraints      0.533                0.9980         99.3                         3.8                                    11.6
D1+D2+D3                      0.539                0.9975         99.3                         4.6 

In [15]:
# CELL 31 — D4 deep defense: "can ANY zlib setting reproduce this text page byte-for-byte?" odd-one-out rule; combined detector = LightGBM OR D4 (dev docs, 15 versions + 6 attacks)
import os, re, zlib, hashlib, random, traceback
import numpy as np
import pandas as pd
import pikepdf
import pymupdf as fitz
from pebble import ProcessPool
from concurrent.futures import TimeoutError, as_completed
from tqdm.auto import tqdm


META_FEATS = META if isinstance(META, list) else META_FEATS   # keep the feature-name list under a new name
META = "/kaggle/working/full/meta"                             # restore the folder path the generator and cleanup need
os.makedirs(META, exist_ok=True)

D4_CSV = f"{ATK}/d4_results.csv"
TEXT_OP = re.compile(rb"T[jJ]")
MIN_DEC, MAX_DEC = 300, 400_000
CACHE = {}

def anyzlib_status(raw):
    """1 = some zlib setting reproduces these bytes exactly, 0 = none does, None = not a usable text stream."""
    try:
        dob = zlib.decompressobj()
        dec = dob.decompress(raw)
        body = raw[:len(raw) - len(dob.unused_data)]
    except Exception:
        return None
    if len(body) < 6 or not (MIN_DEC <= len(dec) <= MAX_DEC) or not TEXT_OP.search(dec):
        return None
    cmf, flg = body[0], body[1]
    if cmf & 0x0F != 8:
        return None
    wb, fl = min(15, max(9, (cmf >> 4) + 8)), flg >> 6
    for lv in range(0, 10):
        for st in STRATS:
            if level_flag(lv, st) != fl:
                continue
            for mem in range(1, 10):
                co = zlib.compressobj(lv, zlib.DEFLATED, wb, mem, st)
                if co.compress(dec) + co.flush() == body:
                    return 1
    return 0

def d4_scan(path):
    items = []
    with pikepdf.open(path) as pdf:
        for i in range(len(pdf.pages)):
            for s in page_streams(pdf, i):
                if filter_names(s) != ["/FlateDecode"]:
                    continue
                raw = s.read_raw_bytes()
                h = hashlib.sha1(raw).digest()
                if h not in CACHE:
                    CACHE[h] = anyzlib_status(raw)
                if CACHE[h] is not None:
                    items.append((i, CACHE[h]))
            if len(items) >= 200:
                break
    return items

def d4_decide(items):
    if len(items) < 2:
        return 0, -1, len(items)
    vals = [s for _, s in items]
    maj = 1 if sum(vals) * 2 > len(vals) else 0
    odd_pages = sorted({p for p, s in items if s != maj})
    npages = len({p for p, _ in items})
    flag = int(1 <= len(odd_pages) <= max(1, npages // 3))
    loc = odd_pages[0] if flag and len(odd_pages) == 1 else -1
    return flag, loc, len(items)

def d4_doc(name):
    global CACHE
    CACHE = {}
    fitz.TOOLS.mupdf_display_errors(False)
    spec = SPECS[name]
    src, dspec = os.path.join(SRC, name), spec["dspec"]
    res, errs = [], []
    # ---- the 15 normal versions (regenerated with the same seeded generator) ----
    try:
        _, rows_, err = process((name, spec["family"]))
        if rows_ is None:
            raise RuntimeError(str(err))
        for r in rows_:
            p = os.path.join(OUT, r["out_file"])
            try:
                flag, loc, n = d4_decide(d4_scan(p))
                res.append({"doc": name, "kind": "version", "name": r["variant"], "label": r["label"],
                            "family": spec["family"], "edit_page": r["edit_page"], "d4_flag": flag, "d4_loc": loc, "d4_n": n})
            except Exception as ex:
                errs.append(f"{name} {r['variant']}: {type(ex).__name__}: {str(ex)[:80]}")
    except Exception as ex:
        errs.append(f"{name} versions: {type(ex).__name__}: {str(ex)[:80]}")
    finally:
        cleanup(name[:-4])
    # ---- the 6 attacked versions ----
    try:
        info_b, ids_b = read_meta_plain(src)
    except Exception as ex:
        errs.append(f"{name} read_meta: {type(ex).__name__}: {str(ex)[:80]}")
        return name, res, errs
    for a in ATTACKS:
        rng = random.Random(zlib.crc32((name + a).encode()))
        out = os.path.join(ATK_PDF, f"{name[:-4]}__{a}.pdf")
        try:
            if a in ("A3_hide_traces", "A5c_combined"):
                build_one_fixed(src, dspec, a, rng, info_b, ids_b)
            else:
                build_one(src, dspec, a, rng, {}, None)
            flag, loc, n = d4_decide(d4_scan(out))
            res.append({"doc": name, "kind": "attack", "name": a, "label": 1, "family": spec["family"],
                        "edit_page": dspec["page"], "d4_flag": flag, "d4_loc": loc, "d4_n": n})
        except Exception as ex:
            errs.append(f"{name} {a}: {type(ex).__name__}: {str(ex)[:80]}")
        finally:
            if os.path.exists(out):
                os.remove(out)
    return name, res, errs

# ---------- clear an empty/broken results file from the earlier run ----------
if os.path.exists(D4_CSV) and os.path.getsize(D4_CSV) <= 10:
    os.remove(D4_CSV)
    print("Removed empty results file from the earlier run.")

dev_docs_all = sorted(d for d, s in SPECS.items() if s["split"] == "dev")
prev = pd.read_csv(D4_CSV) if os.path.exists(D4_CSV) else pd.DataFrame()
done = set(prev.groupby("doc").name.nunique().loc[lambda s: s >= 21].index) if len(prev) else set()
todo = [d for d in dev_docs_all if d not in done]
print(f"Dev docs: {len(dev_docs_all)} | already done: {len(done)} | to run: {len(todo)}")

# ---------- quick test on ONE document first (stops here with the exact error if something is wrong) ----------
if todo:
    test_doc = todo[0]
    try:
        _, test_res, test_errs = d4_doc(test_doc)
    except Exception:
        traceback.print_exc()
        raise SystemExit("Quick test crashed - paste the error above to Claude.")
    print(f"Quick test on {test_doc}: {len(test_res)} of 21 results | errors: {len(test_errs)}")
    for e in test_errs[:5]:
        print("   ", e)
    assert len(test_res) >= 15, "Quick test produced too few results - paste the errors above to Claude before running the rest."
    print("Quick test OK - running all documents.\n")

# ---------- run on all dev docs (resumable, saves every 20 docs) ----------
rows = prev.to_dict("records") if len(prev) else []
all_errs = []
with ProcessPool(max_workers=WORKERS) as pool_ex:
    futs = {pool_ex.schedule(d4_doc, args=(d,), timeout=900): d for d in todo}
    for i, fu in enumerate(tqdm(as_completed(futs), total=len(futs), desc="D4 scan (dev docs)")):
        try:
            _, r, e = fu.result()
            rows += r
            all_errs += e
        except TimeoutError:
            all_errs.append(f"{futs[fu]}: Timeout")
        except Exception as ex:
            all_errs.append(f"{futs[fu]}: {type(ex).__name__}: {str(ex)[:80]}")
        if (i + 1) % 20 == 0 and rows:
            pd.DataFrame(rows).drop_duplicates(["doc", "name"], keep="last").to_csv(D4_CSV, index=False)
d4 = pd.DataFrame(rows).drop_duplicates(["doc", "name"], keep="last")
d4.to_csv(D4_CSV, index=False)
print(f"\nD4 rows: {len(d4)} | docs: {d4.doc.nunique()} | errors: {len(all_errs)}")
for e in all_errs[:5]:
    print("   ", e)

# ---------- combine with LightGBM (no-defense model and threshold from CELL 29) ----------
ver = d4[d4.kind == "version"].merge(devf[["orig_file", "variant", "score", "loc_page"]], left_on=["doc", "name"],
                                     right_on=["orig_file", "variant"], how="left")
att = d4[d4.kind == "attack"].merge(dev_atk[["doc", "attack", "score", "loc_page"]], left_on=["doc", "name"],
                                    right_on=["doc", "attack"], how="left")
for t in (ver, att):
    t["lgb_flag"] = (t.score > THR).astype(int)
    t["combined"] = ((t.lgb_flag == 1) | (t.d4_flag == 1)).astype(int)

VORDER = ["G0_original", "G1_resave", "G2_recompress", "G3_note_incr", "G4_linearize", "G5_stamp_honest", "G6_textbox_honest",
          "F1_digit_resave", "F2_digit_recompress", "F3_digit_incr", "F4_value_resave", "F5_cover_replace",
          "F6_digit_linearize", "F7_stamp_fake", "F8_textbox_fake"]
print("\n=== TABLE 7: FLAG RATE PER VERSION (G = false alarms, F = detections), dev ===")
t7 = ver.groupby("name")[["lgb_flag", "d4_flag", "combined"]].mean().mul(100).round(1).reindex(VORDER)
t7.columns = ["LightGBM %", "D4 %", "LightGBM OR D4 %"]
print(t7.to_string())
hon = ver[ver.label == 0]
print(f"\nFalse alarms on ALL honest dev files: LightGBM {hon.lgb_flag.mean() * 100:.1f}% | D4 {hon.d4_flag.mean() * 100:.1f}%"
      f" | combined {hon.combined.mean() * 100:.1f}%")

print("\n=== TABLE 8: ATTACKS CAUGHT (dev) ===")
t8 = att.groupby("name")[["lgb_flag", "d4_flag", "combined"]].mean().mul(100).round(1).reindex(ATTACKS)
t8.columns = ["LightGBM %", "D4 %", "LightGBM OR D4 %"]
print(t8.to_string())

print("\n=== TABLE 9: A2b (feature-aware mimic) CAUGHT, per family ===")
a2b = att[att.name == "A2b_mimic_aware"]
print(a2b.groupby("family")[["lgb_flag", "d4_flag", "combined"]].mean().mul(100).round(1)
      .rename(columns={"lgb_flag": "LightGBM %", "d4_flag": "D4 %", "combined": "Combined %"}).to_string())

print("\n=== TABLE 10: PAGE LOCALIZATION (dev): D4 page, and D4-then-rule combined ===")
lrows = []
for src_df, names in [(ver, ["F1_digit_resave", "F4_value_resave", "F5_cover_replace", "F6_digit_linearize"]),
                      (att, ["A2b_mimic_aware", "A4_decoys", "A5c_combined"])]:
    for nm in names:
        s = src_df[src_df.name == nm]
        ep = s.edit_page.astype(float)
        d4f = s.d4_loc >= 0
        comb = np.where(d4f, s.d4_loc, s.loc_page.fillna(-1))
        lrows.append({"Version": nm, "n": len(s), "D4 gives a page %": round(d4f.mean() * 100, 1),
                      "D4 correct when it gives one %": round((s.d4_loc[d4f] == ep[d4f]).mean() * 100, 1) if d4f.any() else np.nan,
                      "Rule correct overall %": round((s.loc_page.fillna(-1) == ep).mean() * 100, 1),
                      "D4-then-rule correct overall %": round((comb == ep.values).mean() * 100, 1)})
print(pd.DataFrame(lrows).to_string(index=False))
print(f"\nSaved: {D4_CSV}")

Dev docs: 403 | already done: 0 | to run: 403
Quick test on 000010.pdf: 21 of 21 results | errors: 0
Quick test OK - running all documents.



D4 scan (dev docs):   0%|          | 0/403 [00:00<?, ?it/s]


D4 rows: 8463 | docs: 403 | errors: 0

=== TABLE 7: FLAG RATE PER VERSION (G = false alarms, F = detections), dev ===
                     LightGBM %  D4 %  LightGBM OR D4 %
name                                                   
G0_original                 0.0  14.6              14.6
G1_resave                   5.0  14.6              18.9
G2_recompress               7.9   0.0               7.9
G3_note_incr                0.0  14.6              14.6
G4_linearize                2.7  14.4              16.4
G5_stamp_honest             6.5  14.6              20.1
G6_textbox_honest           6.7  14.6              20.1
F1_digit_resave            99.5  68.0              99.5
F2_digit_recompress         8.2   0.0               8.2
F3_digit_incr             100.0  68.0             100.0
F4_value_resave            99.5  68.2              99.5
F5_cover_replace           99.8  14.6              99.8
F6_digit_linearize         89.8  68.0              90.1
F7_stamp_fake               6.5  14.6    

In [16]:
# CELL 32 — D4 numbers (9 per file) for ALL documents: train + dev, 15 versions each, + 6 attacks for the 1,003 attacked docs; then explain the D4 false alarms
import os, zlib, random, traceback
import numpy as np
import pandas as pd
import pymupdf as fitz
from pebble import ProcessPool
from concurrent.futures import TimeoutError, as_completed
from tqdm.auto import tqdm

# ---------- guard: META must be the folder path used by the generator / cleanup (CELL 20) ----------
if not isinstance(META, str):
    META_FEATS = META
META = "/kaggle/working/full/meta"
os.makedirs(META, exist_ok=True)
os.makedirs(OUT, exist_ok=True)

D4F_CSV = f"{ATK}/d4_features_all.csv"
D4_COLS = ["d4_n", "d4_n_repro", "d4_frac_repro", "d4_npages", "d4_n_odd_pages", "d4_odd_frac", "d4_odd_is_repro", "d4_flag", "d4_loc"]

def d4_features(items):
    n = len(items)
    f = dict(zip(D4_COLS, [n, 0, 0.0, 0, 0, 0.0, 0, 0, -1]))
    if n == 0:
        return f
    vals = [s for _, s in items]
    pages = {p for p, _ in items}
    f["d4_n_repro"], f["d4_frac_repro"], f["d4_npages"] = sum(vals), sum(vals) / n, len(pages)
    if n >= 2:
        maj = 1 if sum(vals) * 2 > n else 0
        odd = {p for p, s in items if s != maj}
        f["d4_n_odd_pages"], f["d4_odd_frac"] = len(odd), len(odd) / len(pages)
        f["d4_odd_is_repro"] = int(maj == 0 and len(odd) > 0)
        flag = int(1 <= len(odd) <= max(1, len(pages) // 3))
        f["d4_flag"] = flag
        f["d4_loc"] = min(odd) if flag and len(odd) == 1 else -1
    return f

def d4_doc2(job):
    global CACHE
    CACHE = {}
    name, fam, split = job
    fitz.TOOLS.mupdf_display_errors(False)
    res, errs = [], []
    try:
        _, rws, err = process((name, fam))
        if rws is None:
            raise RuntimeError(str(err))
        for r in rws:
            p = os.path.join(OUT, r["out_file"])
            try:
                res.append({"doc": name, "kind": "version", "name": r["variant"], "label": r["label"], "split": split,
                            "family": fam, **d4_features(d4_scan(p))})
            except Exception as ex:
                errs.append(f"{name} {r['variant']}: {type(ex).__name__}: {str(ex)[:80]}")
    except Exception as ex:
        errs.append(f"{name} versions: {type(ex).__name__}: {str(ex)[:80]}")
    finally:
        cleanup(name[:-4])
    if name in SPECS:
        spec = SPECS[name]
        src, dspec = os.path.join(SRC, name), spec["dspec"]
        try:
            info_b, ids_b = read_meta_plain(src)
        except Exception as ex:
            errs.append(f"{name} read_meta: {type(ex).__name__}: {str(ex)[:80]}")
            return name, res, errs
        for a in ATTACKS:
            rng = random.Random(zlib.crc32((name + a).encode()))
            out = os.path.join(ATK_PDF, f"{name[:-4]}__{a}.pdf")
            try:
                if a in ("A3_hide_traces", "A5c_combined"):
                    build_one_fixed(src, dspec, a, rng, info_b, ids_b)
                else:
                    build_one(src, dspec, a, rng, {}, None)
                res.append({"doc": name, "kind": "attack", "name": a, "label": 1, "split": split, "family": fam,
                            **d4_features(d4_scan(out))})
            except Exception as ex:
                errs.append(f"{name} {a}: {type(ex).__name__}: {str(ex)[:80]}")
            finally:
                if os.path.exists(out):
                    os.remove(out)
    return name, res, errs

# ---------- job list: every document in the full build ----------
g0 = feat_full[feat_full.variant == "G0_original"][["orig_file", "family", "split"]]
JOBS2 = [tuple(x) for x in g0.itertuples(index=False)]
expected = {d: 15 + (6 if d in SPECS else 0) for d, _, _ in JOBS2}
if os.path.exists(D4F_CSV) and os.path.getsize(D4F_CSV) <= 10:
    os.remove(D4F_CSV)
prev = pd.read_csv(D4F_CSV) if os.path.exists(D4F_CSV) else pd.DataFrame()
done = set()
if len(prev):
    cnt = prev.groupby("doc").name.nunique()
    done = {d for d, c in cnt.items() if c >= expected.get(d, 99)}
todo = [j for j in JOBS2 if j[0] not in done]
print(f"Documents: {len(JOBS2)} (train {sum(j[2] == 'train' for j in JOBS2)}, dev {sum(j[2] == 'dev' for j in JOBS2)})"
      f" | attacked docs: {len(SPECS)} | already done: {len(done)} | to run: {len(todo)}")

# ---------- quick test on ONE document first ----------
if todo:
    tj = next((j for j in todo if j[0] in SPECS), todo[0])
    try:
        _, tres, terrs = d4_doc2(tj)
    except Exception:
        traceback.print_exc()
        raise SystemExit("Quick test crashed - paste the first lines of the error above to Claude.")
    print(f"Quick test on {tj[0]}: {len(tres)} of {expected[tj[0]]} results | errors: {len(terrs)}")
    for e in terrs[:5]:
        print("   ", e)
    assert len(tres) >= expected[tj[0]] - 1, "Quick test produced too few results - paste the errors above to Claude."
    print("Quick test OK - running all documents.\n")

# ---------- run (resumable, saves every 50 documents) ----------
rows2 = prev.to_dict("records") if len(prev) else []
errs2 = []
with ProcessPool(max_workers=WORKERS) as pool_ex:
    futs = {pool_ex.schedule(d4_doc2, args=(j,), timeout=900): j[0] for j in todo}
    for i, fu in enumerate(tqdm(as_completed(futs), total=len(futs), desc="D4 numbers (all docs)")):
        try:
            _, r, e = fu.result()
            rows2 += r
            errs2 += e
        except TimeoutError:
            errs2.append(f"{futs[fu]}: Timeout")
        except Exception as ex:
            errs2.append(f"{futs[fu]}: {type(ex).__name__}: {str(ex)[:80]}")
        if (i + 1) % 50 == 0 and rows2:
            pd.DataFrame(rows2).drop_duplicates(["doc", "name"], keep="last").to_csv(D4F_CSV, index=False)
d4f = pd.DataFrame(rows2).drop_duplicates(["doc", "name"], keep="last")
d4f.to_csv(D4F_CSV, index=False)
print(f"\nRows: {len(d4f)} | documents: {d4f.doc.nunique()} | errors: {len(errs2)}")
for e in errs2[:5]:
    print("   ", e)
print("Rows per kind and split:")
print(d4f.groupby(["kind", "split"]).size().to_string())

# ---------- explain the D4 false alarms on untouched originals (all 2,722 documents) ----------
orig = d4f[(d4f.kind == "version") & (d4f.name == "G0_original")].copy()
print(f"\n=== D4 RULE ON UNTOUCHED ORIGINALS (G0), all documents: flagged {orig.d4_flag.mean() * 100:.1f}% ===")
print("\nPer family:")
print(orig.groupby("family").agg(docs=("d4_flag", "size"), flagged_pct=("d4_flag", lambda s: round(s.mean() * 100, 1)),
                                 median_text_streams=("d4_n", "median"),
                                 median_share_zlib_repro=("d4_frac_repro", "median")).to_string())
fl = orig[orig.d4_flag == 1]
print("\nAmong FLAGGED originals - is the odd page the zlib-reproducible one? (1 = yes)")
print(fl.d4_odd_is_repro.value_counts(normalize=True).mul(100).round(1).to_string())
orig["n_bucket"] = pd.cut(orig.d4_n, [-1, 1, 3, 6, 12, 1000], labels=["0-1", "2-3", "4-6", "7-12", "13+"])
print("\nFlagged % by number of usable text streams in the document:")
print(orig.groupby("n_bucket", observed=True).d4_flag.agg(["size", lambda s: round(s.mean() * 100, 1)])
      .rename(columns={"size": "docs", "<lambda_0>": "flagged_pct"}).to_string())
print(f"\nSaved: {D4F_CSV}")

Documents: 2722 (train 2319, dev 403) | attacked docs: 1003 | already done: 0 | to run: 2722
Quick test on 000010.pdf: 21 of 21 results | errors: 0
Quick test OK - running all documents.



D4 numbers (all docs):   0%|          | 0/2722 [00:00<?, ?it/s]


Rows: 46848 | documents: 2722 | errors: 0
Rows per kind and split:
kind     split
attack   dev       2418
         train     3600
version  dev       6045
         train    34785

=== D4 RULE ON UNTOUCHED ORIGINALS (G0), all documents: flagged 15.8% ===

Per family:
                           docs  flagged_pct  median_text_streams  median_share_zlib_repro
family                                                                                    
acrobat distiller for       620         22.7                  9.0                      0.0
acrobat distiller windows  1511         17.2                 13.0                      0.0
acrobat pdfwriter for       416          4.6                  7.0                      1.0
adobe pdf library            70          5.7                 10.0                      0.0
corel pdf engine            105          5.7                 13.0                      1.0

Among FLAGGED originals - is the odd page the zlib-reproducible one? (1 = yes)
d4_odd_is_repro


In [17]:
# CELL 33 — Learned D4: LightGBM + 8 D4 numbers (with / without attack training) vs baseline; clean accuracy, false alarms, attacks, per family, localization, 95% CIs
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupKFold
from tqdm.auto import tqdm

d4f = pd.read_csv(D4F_CSV)
D4_FEATS = ["d4_n", "d4_n_repro", "d4_frac_repro", "d4_npages", "d4_n_odd_pages", "d4_odd_frac", "d4_odd_is_repro", "d4_flag"]
FULLD4 = FULL + D4_FEATS

# ---------- join D4 numbers to the existing features ----------
v = d4f[d4f.kind == "version"][["doc", "name"] + D4_FEATS + ["d4_loc"]].rename(columns={"doc": "orig_file", "name": "variant"})
ff = feat_full.drop(columns=[c for c in D4_FEATS + ["d4_loc"] if c in feat_full.columns]).merge(v, on=["orig_file", "variant"], how="left")
a4 = d4f[d4f.kind == "attack"][["doc", "name"] + D4_FEATS + ["d4_loc"]].rename(columns={"name": "attack"})
aa = atk.drop(columns=[c for c in D4_FEATS + ["d4_loc"] if c in atk.columns]).merge(a4, on=["doc", "attack"], how="left")
print("Version rows missing D4:", int(ff[D4_FEATS].isna().any(axis=1).sum()), "of", len(ff),
      "| attack rows missing D4:", int(aa[D4_FEATS].isna().any(axis=1).sum()), "of", len(aa))
assert ff[D4_FEATS].isna().any(axis=1).sum() == 0 and aa[D4_FEATS].isna().any(axis=1).sum() == 0, "D4 numbers missing - tell Claude"

tr_b = ff[ff.split == "train"].copy()
adv = aa[aa.split == "train"].copy()
adv["label"], adv["orig_file"] = 1, adv["doc"]
devv = ff[ff.split == "dev"].reset_index(drop=True)
deva = aa[aa.split == "dev"].reset_index(drop=True)
NO_EV = ["F2_digit_recompress", "F7_stamp_fake", "F8_textbox_fake", "G2_recompress", "G5_stamp_honest", "G6_textbox_honest"]
EVm = ~devv.variant.isin(NO_EV).values
HON = devv.label.values == 0
G0m = (devv.variant == "G0_original").values
PKm = (devv.variant == "G1_resave").values & devv.orig_file.isin(pk_docs).values
F1m = (devv.variant == "F1_digit_resave").values

MODELS = [("Baseline (no defense)", FULL, False), ("D1 attack training", FULL, True),
          ("D4-learned (no attack training)", FULLD4, False), ("D4 + D1", FULLD4, True)]

def train_eval(cols, use_adv):
    train = pd.concat([tr_b[cols + ["label", "orig_file"]], adv[cols + ["label", "orig_file"]]], ignore_index=True) \
        if use_adv else tr_b[cols + ["label", "orig_file"]].reset_index(drop=True)
    oof = np.zeros(len(train))
    for a, b in GroupKFold(n_splits=5).split(train, train.label, train.orig_file):
        oof[b] = lgb.LGBMClassifier(**PARAMS).fit(train.iloc[a][cols], train.iloc[a].label).predict_proba(train.iloc[b][cols])[:, 1]
    thr = float(np.quantile(oof[train.label.values == 0], 0.95))
    m = lgb.LGBMClassifier(**PARAMS).fit(train[cols], train.label)
    return m, thr, m.predict_proba(devv[cols])[:, 1], m.predict_proba(deva[cols])[:, 1]

out_rows, fam_rows, store = [], {}, {}
for name, cols, use_adv in tqdm(MODELS, desc="Training models"):
    m, thr, sv, sa = train_eval(cols, use_adv)
    store[name] = (m, thr, sv, sa, cols)
    r = {"Model": name, "Thr": round(thr, 3),
         "Clean AUC (evid)": round(roc_auc_score(devv.label.values[EVm], sv[EVm]), 4),
         "F1 caught %": round((sv[F1m] > thr).mean() * 100, 1),
         "FA all honest %": round((sv[HON] > thr).mean() * 100, 1),
         "FA originals %": round((sv[G0m] > thr).mean() * 100, 1),
         "FA pikepdf re-save %": round((sv[PKm] > thr).mean() * 100, 1)}
    for at in ATTACKS:
        mk = (deva.attack == at).values
        r[at] = round((sa[mk] > thr).mean() * 100, 1)
    out_rows.append(r)
    mk = (deva.attack == "A2b_mimic_aware").values
    fam_rows[name] = pd.Series(sa[mk] > thr, index=deva.family[mk]).groupby(level=0).mean().mul(100).round(1)

res = pd.DataFrame(out_rows).set_index("Model")
print("\n=== TABLE 11: CLEAN ACCURACY AND FALSE ALARMS (dev, 403 docs) ===")
print(res[["Thr", "Clean AUC (evid)", "F1 caught %", "FA all honest %", "FA originals %", "FA pikepdf re-save %"]].to_string())
print("\n=== TABLE 12: % OF EACH ATTACK CAUGHT (dev) ===")
print(res[ATTACKS].to_string())
print("\n=== TABLE 13: A2b (feature-aware mimic) CAUGHT %, per family ===")
print(pd.DataFrame(fam_rows).to_string())

# ---------- how much the D4 numbers matter inside the learned model ----------
m4, _, _, _, c4 = store["D4-learned (no attack training)"]
imp = pd.Series(m4.booster_.feature_importance("gain"), index=c4)
imp = (imp / imp.sum() * 100).sort_values(ascending=False)
print("\n=== TABLE 14: TOP 10 FEATURES in D4-learned model (% of gain) ===")
print(imp.head(10).round(1).to_string())
print(f"Total share of the 8 D4 numbers: {imp[D4_FEATS].sum():.1f}%")

# ---------- 95% CIs (bootstrap over dev documents): A2b caught and false alarms on originals ----------
print("\n=== TABLE 15: 95% CONFIDENCE INTERVALS (1,000 bootstrap resamples over dev documents) ===")
rng = np.random.default_rng(0)
docs = devv.orig_file.unique()
bs = [rng.choice(docs, size=len(docs), replace=True) for _ in range(1000)]
a2b_idx = deva[deva.attack == "A2b_mimic_aware"].reset_index().set_index("doc")["index"]
g0_idx = devv[G0m].reset_index().set_index("orig_file")["index"]
hon_by_doc = devv[HON].groupby("orig_file").apply(lambda g: g.index.values)
ci_rows = []
for name in store:
    _, thr, sv, sa, _ = store[name]
    c_a2b, c_fa = [], []
    for d in bs:
        ia = a2b_idx.reindex(d).dropna().astype(int).values
        ih = np.concatenate([hon_by_doc[x] for x in d if x in hon_by_doc.index])
        c_a2b.append((sa[ia] > thr).mean() * 100)
        c_fa.append((sv[ih] > thr).mean() * 100)
    ci_rows.append({"Model": name,
                    "A2b caught % [95% CI]": f"{np.mean(c_a2b):.1f} [{np.percentile(c_a2b, 2.5):.1f}, {np.percentile(c_a2b, 97.5):.1f}]",
                    "FA all honest % [95% CI]": f"{np.mean(c_fa):.1f} [{np.percentile(c_fa, 2.5):.1f}, {np.percentile(c_fa, 97.5):.1f}]"})
print(pd.DataFrame(ci_rows).to_string(index=False))

# ---------- localization under attack (D4 page, rule page, D4-then-rule) ----------
print("\n=== TABLE 16: PAGE LOCALIZATION (dev) ===")
lrows = []
for df_, names, key in [(devv, ["F1_digit_resave", "F4_value_resave", "F6_digit_linearize"], "variant"),
                        (deva, ["A0_plain", "A2b_mimic_aware", "A4_decoys", "A5c_combined"], "attack")]:
    for nm in names:
        s = df_[df_[key] == nm]
        ep = s.edit_page.astype(float).values
        d4l = s.d4_loc.values
        rl = s.loc_page.fillna(-1).values
        comb = np.where(d4l >= 0, d4l, rl)
        gives = d4l >= 0
        lrows.append({"Version": nm, "n": len(s),
                      "D4 gives page %": round(gives.mean() * 100, 1),
                      "D4 correct when given %": round((d4l[gives] == ep[gives]).mean() * 100, 1) if gives.any() else np.nan,
                      "Rule correct %": round((rl == ep).mean() * 100, 1),
                      "D4-then-rule correct %": round((comb == ep).mean() * 100, 1)})
print(pd.DataFrame(lrows).to_string(index=False))

# ---------- save everything for final tables/figures ----------
save_v = devv[["orig_file", "variant", "label", "family", "edit_page", "loc_page", "d4_loc"]].copy()
save_a = deva[["doc", "attack", "family", "edit_page", "loc_page", "d4_loc"]].copy()
for name in store:
    _, thr, sv, sa, _ = store[name]
    save_v[name], save_a[name] = sv, sa
save_v.to_csv(f"{ATK}/final_dev_versions_scores.csv", index=False)
save_a.to_csv(f"{ATK}/final_dev_attacks_scores.csv", index=False)
res.to_csv(f"{ATK}/final_defense_table.csv")
print(f"\nSaved: final_dev_versions_scores.csv, final_dev_attacks_scores.csv, final_defense_table.csv in {ATK}")

Version rows missing D4: 0 of 40830 | attack rows missing D4: 0 of 6018


Training models:   0%|          | 0/4 [00:00<?, ?it/s]


=== TABLE 11: CLEAN ACCURACY AND FALSE ALARMS (dev, 403 docs) ===
                                   Thr  Clean AUC (evid)  F1 caught %  FA all honest %  FA originals %  FA pikepdf re-save %
Model                                                                                                                       
Baseline (no defense)            0.531            0.9981         99.5              4.1             0.0                  11.6
D1 attack training               0.542            0.9973         99.3              4.9             0.0                  25.6
D4-learned (no attack training)  0.529            0.9981         99.3              3.7             0.0                  11.6
D4 + D1                          0.543            0.9976         99.3              4.5             0.0                  27.9

=== TABLE 12: % OF EACH ATTACK CAUGHT (dev) ===
                                 A0_plain  A2a_mimic_simple  A2b_mimic_aware  A3_hide_traces  A4_decoys  A5c_combined
Model           

/tmp/ipykernel_48/2815357532.py:87: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  hon_by_doc = devv[HON].groupby("orig_file").apply(lambda g: g.index.values)


                          Model A2b caught % [95% CI] FA all honest % [95% CI]
          Baseline (no defense)     15.4 [12.2, 18.9]           4.1 [3.2, 5.0]
             D1 attack training     30.3 [26.1, 34.7]           4.9 [3.8, 6.0]
D4-learned (no attack training)     15.4 [12.2, 19.1]           3.7 [2.8, 4.6]
                        D4 + D1     87.8 [84.6, 90.8]           4.5 [3.5, 5.5]

=== TABLE 16: PAGE LOCALIZATION (dev) ===
           Version   n  D4 gives page %  D4 correct when given %  Rule correct %  D4-then-rule correct %
   F1_digit_resave 403             58.3                     99.1            83.4                    88.8
   F4_value_resave 403             58.6                     99.2            83.4                    88.8
F6_digit_linearize 403             58.3                     99.1            76.7                    82.1
          A0_plain 403             58.3                     99.1            76.7                    82.1
   A2b_mimic_aware 403             58

In [18]:
# CELL 33b — FINAL defense: add matched honest pikepdf re-saves (no edit) for the 1,003 attacked docs, retrain, final tables with 95% CIs, write FINAL_RESULTS.txt
import os
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupKFold
from pebble import ProcessPool
from concurrent.futures import TimeoutError, as_completed
from tqdm.auto import tqdm

# ---------- 1. honest pikepdf re-save (no edit) for every attacked document ----------
H_CSV = f"{ATK}/honest_pikepdf_resaves.csv"

def honest_job(name):
    global CACHE
    CACHE = {}
    src = os.path.join(SRC, name)
    out = os.path.join(ATK_PDF, f"{name[:-4]}__H_pikepdf_resave.pdf")
    try:
        run_tool("pikepdf", src, out, None, "resave")
        f = extract_features(out)
        f.update(d4_features(d4_scan(out)))
        f.update({"doc": name, "split": SPECS[name]["split"], "family": SPECS[name]["family"]})
        return f, None
    except Exception as ex:
        return None, f"{name}: {type(ex).__name__}: {str(ex)[:60]}"
    finally:
        if os.path.exists(out):
            os.remove(out)

if os.path.exists(H_CSV) and os.path.getsize(H_CSV) > 10:
    HS = pd.read_csv(H_CSV)
else:
    hrows, herrs = [], []
    with ProcessPool(max_workers=WORKERS) as pool_ex:
        futs = {pool_ex.schedule(honest_job, args=(d,), timeout=300): d for d in SPECS}
        for fu in tqdm(as_completed(futs), total=len(futs), desc="Honest pikepdf re-saves"):
            try:
                f, e = fu.result()
                if f:
                    hrows.append(f)
                if e:
                    herrs.append(e)
            except Exception as ex:
                herrs.append(f"{futs[fu]}: {type(ex).__name__}")
    HS = pd.DataFrame(hrows)
    HS.to_csv(H_CSV, index=False)
    print("Honest re-saves built:", len(HS), "| errors:", len(herrs), herrs[:3])
HS["label"], HS["orig_file"] = 0, HS["doc"]
hs_tr = HS[HS.split == "train"]
hs_dev = HS[HS.split == "dev"].reset_index(drop=True)
print(f"Honest pikepdf re-saves: train {len(hs_tr)} | dev {len(hs_dev)}")

# ---------- 2. train the final comparison ----------
def train_parts(cols, parts):
    train = pd.concat([p[cols + ["label", "orig_file"]] for p in parts], ignore_index=True)
    oof = np.zeros(len(train))
    for a, b in GroupKFold(n_splits=5).split(train, train.label, train.orig_file):
        oof[b] = lgb.LGBMClassifier(**PARAMS).fit(train.iloc[a][cols], train.iloc[a].label).predict_proba(train.iloc[b][cols])[:, 1]
    thr = float(np.quantile(oof[train.label.values == 0], 0.95))
    m = lgb.LGBMClassifier(**PARAMS).fit(train[cols], train.label)
    return thr, m.predict_proba(devv[cols])[:, 1], m.predict_proba(deva[cols])[:, 1], m.predict_proba(hs_dev[cols])[:, 1]

FINAL_MODELS = [
    ("Baseline (no defense)", FULL, [tr_b]),
    ("D4 + attack training", FULLD4, [tr_b, adv]),
    ("Attack training + honest re-saves, NO D4", FULL, [tr_b, adv, hs_tr]),
    ("FINAL: D4 + attack training + honest re-saves", FULLD4, [tr_b, adv, hs_tr]),
]
HONm = devv.label.values == 0
G0m = (devv.variant == "G0_original").values
EVm = ~devv.variant.isin(["F2_digit_recompress", "F7_stamp_fake", "F8_textbox_fake", "G2_recompress",
                          "G5_stamp_honest", "G6_textbox_honest"]).values
F1m = (devv.variant == "F1_digit_resave").values
A2Bm = (deva.attack == "A2b_mimic_aware").values

rows, fam_a2b, fam_fa, store = [], {}, {}, {}
for name, cols, parts in tqdm(FINAL_MODELS, desc="Final models"):
    thr, sv, sa, sh = train_parts(cols, parts)
    store[name] = (thr, sv, sa, sh)
    r = {"Model": name, "Clean AUC (evid)": round(roc_auc_score(devv.label.values[EVm], sv[EVm]), 4),
         "F1 caught %": round((sv[F1m] > thr).mean() * 100, 1),
         "FA all honest %": round((sv[HONm] > thr).mean() * 100, 1),
         "FA originals %": round((sv[G0m] > thr).mean() * 100, 1),
         "FA honest pikepdf re-save %": round((sh > thr).mean() * 100, 1)}
    for at in ATTACKS:
        r[at] = round((sa[(deva.attack == at).values] > thr).mean() * 100, 1)
    rows.append(r)
    fam_a2b[name] = pd.Series(sa[A2Bm] > thr, index=deva.family[A2Bm].values).groupby(level=0).mean().mul(100).round(1)
    fam_fa[name] = pd.Series(sh > thr, index=hs_dev.family.values).groupby(level=0).mean().mul(100).round(1)

fin = pd.DataFrame(rows).set_index("Model")
print("\n=== FINAL TABLE A: CLEAN ACCURACY AND FALSE ALARMS (dev, 403 docs) ===")
print(fin[["Clean AUC (evid)", "F1 caught %", "FA all honest %", "FA originals %", "FA honest pikepdf re-save %"]].to_string())
print("\n=== FINAL TABLE B: % OF EACH ATTACK CAUGHT (dev) ===")
print(fin[ATTACKS].to_string())
print("\n=== FINAL TABLE C: A2b CAUGHT %, per family ===")
print(pd.DataFrame(fam_a2b).to_string())
print("\n=== FINAL TABLE D: FALSE ALARMS ON HONEST pikepdf RE-SAVES %, per family ===")
print(pd.DataFrame(fam_fa).to_string())

# ---------- 3. 95% CIs (bootstrap over dev documents) ----------
rng = np.random.default_rng(0)
docs = devv.orig_file.unique()
draws = [rng.choice(docs, size=len(docs), replace=True) for _ in range(1000)]
a2b_pos = pd.Series(np.where(A2Bm)[0], index=deva.doc[A2Bm].values)
hon_pos = devv[HONm].groupby("orig_file").indices
hs_pos = pd.Series(np.arange(len(hs_dev)), index=hs_dev.doc.values)
ci = []
for name, (thr, sv, sa, sh) in store.items():
    c1, c2, c3 = [], [], []
    for d in draws:
        ia = a2b_pos.reindex(d).dropna().astype(int).values
        ih = np.concatenate([devv.index[hon_pos[x]].values if False else hon_pos[x] for x in d if x in hon_pos])
        ip = hs_pos.reindex(d).dropna().astype(int).values
        c1.append((sa[ia] > thr).mean() * 100)
        c2.append((sv[HONm][0:0].tolist() and 0) or (sv[np.where(HONm)[0]][np.isin(np.where(HONm)[0], ih)] > thr).mean() * 100 if False else (sv[ih] > thr).mean() * 100)
        c3.append((sh[ip] > thr).mean() * 100)
    f = lambda c: f"{np.mean(c):.1f} [{np.percentile(c, 2.5):.1f}, {np.percentile(c, 97.5):.1f}]"
    ci.append({"Model": name, "A2b caught %": f(c1), "FA all honest %": f(c2), "FA honest pikepdf re-save %": f(c3)})
ci = pd.DataFrame(ci)
print("\n=== FINAL TABLE E: 95% CONFIDENCE INTERVALS (1,000 resamples over dev documents) ===")
print(ci.to_string(index=False))

# ---------- 4. save final outputs ----------
fin.to_csv(f"{ATK}/FINAL_defense_table.csv")
ci.to_csv(f"{ATK}/FINAL_defense_CIs.csv", index=False)
with open(f"{ATK}/FINAL_RESULTS.txt", "w") as fh:
    fh.write("FINAL ATTACK/DEFENSE RESULTS (dev set, 403 documents, thresholds set on train docs for ~5% false alarms)\n\n")
    fh.write(fin.to_string() + "\n\n")
    fh.write("A2b caught per family:\n" + pd.DataFrame(fam_a2b).to_string() + "\n\n")
    fh.write("False alarms on honest pikepdf re-saves per family:\n" + pd.DataFrame(fam_fa).to_string() + "\n\n")
    fh.write(ci.to_string(index=False) + "\n")
print(f"\nSaved: FINAL_defense_table.csv, FINAL_defense_CIs.csv, FINAL_RESULTS.txt in {ATK}")

Honest pikepdf re-saves:   0%|          | 0/1003 [00:00<?, ?it/s]

Honest re-saves built: 1003 | errors: 0 []
Honest pikepdf re-saves: train 600 | dev 403


Final models:   0%|          | 0/4 [00:00<?, ?it/s]


=== FINAL TABLE A: CLEAN ACCURACY AND FALSE ALARMS (dev, 403 docs) ===
                                               Clean AUC (evid)  F1 caught %  FA all honest %  FA originals %  FA honest pikepdf re-save %
Model                                                                                                                                     
Baseline (no defense)                                    0.9981         99.5              4.1             0.0                         12.4
D4 + attack training                                     0.9976         99.3              4.5             0.0                         30.8
Attack training + honest re-saves, NO D4                 0.9976         99.3              3.8             0.0                         20.8
FINAL: D4 + attack training + honest re-saves            0.9978         99.3              3.7             0.0                         25.6

=== FINAL TABLE B: % OF EACH ATTACK CAUGHT (dev) ===
                                        

In [19]:
# CELL 33c — Correct the "FA all honest" 95% CI (Table E used wrong row positions)
hon_idx = np.where(HONm)[0]
pos_by_doc = pd.Series(hon_idx).groupby(devv.orig_file.values[hon_idx]).apply(lambda s: s.values)
fix_rows = []
for name, (thr, sv, sa, sh) in store.items():
    c = []
    for d in draws:
        ih = np.concatenate([pos_by_doc[x] for x in d if x in pos_by_doc.index])
        c.append((sv[ih] > thr).mean() * 100)
    fix_rows.append({"Model": name, "FA all honest % [95% CI]": f"{np.mean(c):.1f} [{np.percentile(c, 2.5):.1f}, {np.percentile(c, 97.5):.1f}]"})
fix = pd.DataFrame(fix_rows)
print(fix.to_string(index=False))
ci["FA all honest %"] = fix["FA all honest % [95% CI]"].values
ci.to_csv(f"{ATK}/FINAL_defense_CIs.csv", index=False)
with open(f"{ATK}/FINAL_RESULTS.txt", "a") as fh:
    fh.write("\nCORRECTED 'FA all honest' 95% CIs:\n" + fix.to_string(index=False) + "\n")
print("Fixed and saved.")

                                        Model FA all honest % [95% CI]
                        Baseline (no defense)           4.1 [3.2, 5.0]
                         D4 + attack training           4.5 [3.5, 5.5]
     Attack training + honest re-saves, NO D4           3.8 [2.9, 4.7]
FINAL: D4 + attack training + honest re-saves           3.7 [2.8, 4.6]
Fixed and saved.
